# 머신러닝 파일별 6 · 9 결정 트리 · 질문 · 불순도 · 가지치기 · 흔들림

**한 줄 요약** · 모델이 왜 그렇게 판단했는지 사람이 읽을 수 있을까? — 질문 경로 읽기 → Gini로 좋은 질문 고르기 → 깊이 · 가지치기 → 봉투 1회 열기 → 중요도 · 흔들림까지 Iris로

- **원본** · `6.머닝러신/04_분류모델/9_decision_v2.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-1 (4장 깊이 · 과적합) · 파일별 7(앙상블)로 이어짐

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 트리 상자(질문 · gini · samples · value)를 읽고 꽃 한 송이의 경로를 손으로 따라간다
2. 두 후보 질문 중 무엇이 좋은지 Count → Rate → 가중 Gini로 말한다
3. `max_depth`를 바꿀 때 외운 문제 점수와 처음 보는 문제 점수가 따로 노는 것을 설명한다
4. 선택이 끝나기 전에 최종 평가 데이터를 열지 않는 이유와 '계약'을 말한다
5. `feature_importances_` 0을 '쓸모없다'로 읽으면 틀리는 이유와 트리의 흔들림을 설명한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **모델의 판단 근거를 보고서에 적어야 할 때** · 질문 경로 · 트리 그림 (1장)
- **트리가 train 100%인데 실제 성능이 낮을 때** · 깊이 · 가지치기 (3 · 4장)
- **하이퍼파라미터 결과를 공유할 때** · 후보 · 지표 · CV '계약'을 함께 (4-3)
- **중요도 막대를 보고 피처를 지우려 할 때** · 0 ≠ 쓸모없음 · seed 흔들림 (5-2 · 6-1)

## 학습 지도

<img src="images/pfm6_map.png" alt="9 결정 트리 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (전체 실행 30초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>Iris 준비 · 원본 목표</td></tr>
<tr><td>1</td><td>질문 경로 읽기</td><td>트리 상자 하나는 무엇을 말하고, 꽃 한 송이는 어느 잎으로 가는가?</td></tr>
<tr><td>2</td><td>좋은 질문 고르기</td><td>트리는 왜 하필 그 첫 질문을 골랐는가?</td></tr>
<tr><td>3</td><td>질문 횟수 제한</td><td>질문을 몇 번까지 허용해야 하고, 깊이는 경계를 어떻게 바꾸는가?</td></tr>
<tr><td>4</td><td>가지치기와 후보 비교</td><td>가지치기는 성능을 올리는가? 후보는 어떤 '계약'으로 비교하고, 계약이 바뀌면 무엇이 바뀌는가?</td></tr>
<tr><td>5</td><td>봉투 열기와 해석</td><td>봉인한 30개를 한 번 열면 몇 개를 맞히고, 선택된 트리는 무엇을 말할 수 있는가?</td></tr>
<tr><td>6</td><td>흔들림과 보조 실험</td><td>데이터를 그대로 두고 seed만 바꿔도 규칙이 바뀌는가? 기준 · 결측 · 불균형은 트리에서 어떻게 다루는가?</td></tr>
<tr><td>7</td><td>종합 실습</td><td>한 그루가 흔들린다면 여러 그루의 다수결은 무엇을 바꾸고, 세 모델은 같은 계약에서 어떻게 비교되는가?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

Iris를 불러오기만 하고, 30개 봉투(FINAL_HOLDOUT) 봉인은 1-1에서 함. 원본의 긴 도우미 함수는 쓰는 섹션 안에 짧게 풀어 씀. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · numpy · pandas · scikit-learn (원본의 matplotlib 그림은 images/로)
- **seed** · `RANDOM_STATE = 42` · CV = StratifiedKFold(5, shuffle=True, 42)
- **데이터** · Iris 150 → 개발 120 · 봉투 30 (품종 비율 유지)
- **원본 형식** · Prediction 정답의 접힌 블록(details 태그) → 각 섹션의 '결과' 줄로

**실행 흐름**

1. 모델 · 평가 도구 불러오기
2. seed · CV 계약 정하기
3. Iris 불러오기

In [1]:
# 준비: Iris 30개를 봉투에 봉인 (원본 셀 3 · 선택이 끝난 뒤 한 번만)
# 왜 · 모델을 고르는 동안 최종 평가 데이터를 볼 수 없게 막아 낙관적 점수를 피하기 위해
import numpy as np                                                      # 배열
import pandas as pd                                                     # 표
from sklearn.datasets import load_iris                                  # Iris
from sklearn.metrics import accuracy_score, confusion_matrix            # 지표
from sklearn.model_selection import GridSearchCV, StratifiedKFold       # 탐색 · 등분
from sklearn.model_selection import cross_val_predict, cross_val_score  # CV
from sklearn.model_selection import train_test_split                    # 분할
from sklearn.tree import DecisionTreeClassifier, export_text            # 트리 · 글자로 보기

# 왜 · 분할 · CV · 모델 seed를 한 곳에서 정해 같은 환경에서 같은 결과를 얻기 위해
RANDOM_STATE = 42                                                                # 원본과 같은 seed
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)        # 오늘의 CV 계약
iris = load_iris()                                                               # 150송이
X, y = iris.data, iris.target                                                    # 측정값 4개 · 품종
feature_names, target_names = list(iris.feature_names), list(iris.target_names)  # 이름표
DT, RS = DecisionTreeClassifier, RANDOM_STATE  # 짧은 이름 (코드 폭 줄이기)
print("Iris:", X.shape, "· 품종:", target_names)                                 # 150 × 4

Iris: (150, 4) · 품종: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


### 원본 목표 · 오늘의 흐름

- **오늘의 질문** · 모델이 왜 그렇게 판단했는지 사람이 읽을 수 있을까?
- 로지스틱 회귀와 KNN은 판단 근거를 꺼내 보기 어려웠음 → 결정 트리는 다름

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">알고리즘</th><th style="text-align: left;">판단 방식</th><th style="text-align: left;">경계 모양</th><th style="text-align: left;">근거를 읽을 수 있나</th></tr>
</thead>
<tbody>
<tr><td>로지스틱 회귀</td><td>점수 → 확률</td><td>직선</td><td>계수 12개</td></tr>
<tr><td>KNN</td><td>가까운 이웃 다수결</td><td>곡선</td><td>어려움</td></tr>
<tr><td><strong>결정 트리</strong></td><td><strong>예 / 아니오 질문 반복</strong></td><td><strong>축에 평행한 계단</strong></td><td><strong>그림 한 장</strong></td></tr>
</tbody></table>

- 흐름 · Act 1 질문 경로 → 2 좋은 질문 → 3 질문 횟수 → 4 가지치기 · 후보 비교 → 5 봉투 1회 → 6 해석 → 7 흔들림 → Support A ~ F

**원본 힌트**

- 규칙 하나 · Prediction 칸은 실행하기 전에 채우기
- Support 6개(Gini vs Entropy · 결측 · 불균형 · 계약 · 경계 모양 · 다수결)는 Core가 아님 → 6장 · 7장에 넣음

---
## 1. 질문 경로 읽기

> **이 장의 흐름**
>
> - 평가용 30개를 봉투에 봉인 → 질문을 두 번만 하는 트리로 상자 읽기
> - 그다음 꽃 한 송이가 잎까지 가는 길을 손으로 따라감

<img src="images/pfm6_tree.png" alt="max_depth=2 트리 · 상자마다 질문 · gini · samples · value" width="733">

**파이프라인 위치** · ④ 모델 — 판단 규칙 읽기

### 1-1. `A` 봉투 봉인 · `FinalHoldoutVault` · `open_once` <sub>원본 셀 2 ~ 3</sub>

> 🎯 **선택이 끝나기 전에는 열 수 없는 평가 봉투는 어떻게 만드는가?**

💡 **쉽게 말하면**

- 150개 중 **30개를 봉투에 넣고 봉인** · 나머지 120개(개발)로만 공부 · 선택.
- 선택을 종료(freeze)하기 전에는 열 수 없고, 한 번 열면 다시 못 엶.

**실행 흐름**

1. 개발 120 · 봉투 30 분할
2. 봉투 클래스 (숨김 · 종료 여부 · 연 횟수)
3. 봉인 후 원본 변수 삭제

**먼저 예측** · 선택 종료 전에 open_once()를 부르면? — 답은 코드 실행 결과 아래에

In [2]:
# 왜 · 개발용과 최종 평가용을 처음에 한 번만 나누기 위해
X_development, _Xh, y_development, _yh = train_test_split(  # 개발 120 · 봉투 30
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)


# 왜 · 최종 평가 데이터를 실수로 두 번 쓰는 일을 코드로 막기 위해
class FinalHoldoutVault:                          # 선택이 끝난 뒤 단 한 번만 열 수 있는 평가 봉투
    def __init__(self, X_value, y_value):         # 봉투에 넣기
        self.__X, self.__y = X_value.copy(), y_value.copy()  # 밖에서 못 보게 숨김
        self._frozen, self.open_count = False, 0  # 선택 종료 여부 · 연 횟수

    def freeze_selection(self):  # 선택 종료 선언
        self._frozen = True      # 이제 열 수 있음

    # 왜 · 선택을 끝내기 전에는 열 수 없고, 연 뒤에는 다시 열 수 없게 하기 위해
    def open_once(self):                                         # 한 번만 열기
        if not self._frozen:                                     # 선택이 안 끝났으면
            raise RuntimeError("모델 선택을 먼저 종료해야 함.")  # 막음
        if self.open_count:                                      # 이미 열었으면
            raise RuntimeError("FINAL_HOLDOUT은 이미 열었음.")   # 막음
        self.open_count += 1                                     # 연 횟수 기록
        return self.__X.copy(), self.__y.copy()                  # 꺼내 줌


# 왜 · 원래 변수를 지워 봉투 밖에서 최종 데이터에 손대지 못하게 하기 위해
FINAL_HOLDOUT = FinalHoldoutVault(_Xh, _yh)        # 봉인
del _Xh, _yh                                       # 원본 변수도 지움 (실수로 못 쓰게)
print(f"DEVELOPMENT_POOL {len(X_development)}개 / FINAL_HOLDOUT 30개 봉인")  # 120 / 30
print(f"open count = {FINAL_HOLDOUT.open_count}")  # 0 확인
Xd, yd = X_development, y_development              # 개발 120개의 짧은 이름
try:                                               # 선택 종료 전에 열어 보기
    FINAL_HOLDOUT.open_once()                      # 아직 freeze 안 함
except RuntimeError as e:                          # 막힘
    print("열기 시도:", e)                         # 안내 문구

DEVELOPMENT_POOL 120개 / FINAL_HOLDOUT 30개 봉인
open count = 0
열기 시도: 모델 선택을 먼저 종료해야 함.


**결과** · 예측 정답은 RuntimeError — 선택을 먼저 종료해야 함

- `DEVELOPMENT_POOL 120개 / FINAL_HOLDOUT 30개 봉인` — 개발 120 · 봉투 30
- `open count = 0` — 아직 한 번도 안 열림
- `열기 시도: 모델 선택을 먼저 종료해야 함.` — 선택 종료 전에는 봉투가 막아 줌

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>개발 120개</td><td>학습 · CV · 탐색 전부</td><td>선택할 때</td></tr>
<tr><td>봉투 30개</td><td>선택 종료 뒤 1회</td><td>보고할 때 (5-1)</td></tr>
<tr><td><code>__X</code> (이름 앞 밑줄 두 개)</td><td>밖에서 바로 못 꺼내게 숨김</td><td>실수 방지</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 봉투는 실수를 막는 장치일 뿐 · 결과를 보고 다시 고르지 않는 규율은 사람이 지킴

> ✅ **한 줄 정리** · 평가 데이터는 처음부터 봉투에 봉인하고, 선택을 종료한 뒤에만 한 번 열 수 있게 만든다.

**딥러닝 코드에서는**

- test loader를 함수 안에 숨기고 마지막에 한 번만 부르는 습관

### 1-2. `A` 얕은 트리 · 상자 읽기 · `max_depth=2` · `export_text` <sub>원본 셀 4 ~ 8</sub>

> 🎯 **질문을 두 번만 하는 트리의 첫 질문은 무엇이고, 상자의 value는 무엇을 세는가?**

💡 **쉽게 말하면**

- 결정 트리 = **스무고개** · 예 / 아니오 질문으로 분류.
- 루트 = 첫 질문 · 잎 = 답을 내는 자리 · 깊이 = 한 경로의 질문 횟수.

**실행 흐름**

1. max_depth=2 트리 학습
2. 트리를 글자로 출력
3. 잎마다 개수(value)

**먼저 예측** · 첫 질문은 어떤 피처일까? — 답은 코드 실행 결과 아래에

In [3]:
# 왜 · 깊이 2짜리 트리로 질문 구조를 눈으로 읽을 수 있게 하기 위해
shallow_tree = DT(max_depth=2, random_state=RS)                                  # 질문 최대 2번
shallow_tree.fit(Xd, yd)                                                         # 개발 120개로만
names_short = [n.replace(" (cm)", "") for n in feature_names]                    # 짧은 이름
print(export_text(shallow_tree, feature_names=names_short, show_weights=True))   # 트리를 글자로
t = shallow_tree.tree_                                                           # 내부 구조
leaves = [i for i in range(t.node_count) if t.children_left[i] == -1]            # 잎 번호
for i in leaves:                                                                 # 잎마다
    counts = (t.value[i][0] * t.n_node_samples[i]).round().astype(int).tolist()  # 비율 × 개수
    # 상자 정보
    print(f"잎 {i}: gini {t.impurity[i]:.4f} · samples {t.n_node_samples[i]} · value {counts}")

|--- petal length <= 2.45
|   |--- weights: [40.00, 0.00, 0.00] class: 0
|--- petal length >  2.45
|   |--- petal width <= 1.65
|   |   |--- weights: [0.00, 39.00, 3.00] class: 1
|   |--- petal width >  1.65
|   |   |--- weights: [0.00, 1.00, 37.00] class: 2

잎 1: gini 0.0000 · samples 40 · value [40, 0, 0]
잎 3: gini 0.1327 · samples 42 · value [0, 39, 3]
잎 4: gini 0.0512 · samples 38 · value [0, 1, 37]


**결과** · 예측 정답은 petal length (≤ 2.45)

- `|--- petal length <= 2.45` — 첫 질문(루트) · 꽃잎 길이
- `잎 1: gini 0.0000 · samples 40 · value [40, 0, 0]` — setosa만 40개 → gini 0
- `잎 3: gini 0.1327 · samples 42 · value [0, 39, 3]` — versicolor 39 · virginica 3 섞임
- `잎 4: gini 0.0512 · samples 38 · value [0, 1, 37]` — virginica 37 · versicolor 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>feature &lt;= 값</code></td><td>이 상자의 질문</td><td><code>petal length &lt;= 2.45</code></td></tr>
<tr><td><code>gini</code> · <code>samples</code></td><td>섞인 정도(0이면 한 종류) · 여기까지 온 개수</td><td>잎의 순도 · 크기</td></tr>
<tr><td><code>value</code> · <code>class</code></td><td>클래스별 개수(알파벳순) · 가장 많은 클래스</td><td><code>[0, 39, 3]</code> → versicolor</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `value` 순서는 setosa · versicolor · virginica (알파벳순)
> - sklearn 버전에 따라 `tree_.value`가 개수가 아닌 비율로 저장됨 → 개수는 비율 × samples

> ✅ **한 줄 정리** · 트리 상자는 질문 · 섞인 정도(gini) · 개수(samples) · 클래스별 개수(value)를 보여 주는 순서 있는 규칙이다.

**딥러닝 코드에서는**

- 딥러닝은 이런 규칙을 꺼낼 수 없음 → 설명이 필요하면 트리 · 규칙 모델을 함께 둠

### 1-3. `A` 꽃 한 송이 따라가기 · `decision_path` · `apply` · `predict_proba` <sub>원본 셀 8 ~ 11</sub>

> 🎯 **개발 데이터 세 번째 꽃(`index=2`)은 어느 길로 가고, 확률은 무엇인가?**

💡 **쉽게 말하면**

- 예측은 100% 확신이 아니라 **잎에서 가장 많은 클래스**.
- predict_proba = 잎의 비율 (손으로 센 숫자).

**실행 흐름**

1. 꽃 치수
2. 지나간 질문마다 관측값 비교
3. 도착 잎 · 예측 · 확률

**먼저 예측** · 도착한 잎에서 가장 큰 비율은 몇 %? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 꽃 한 송이가 어떤 질문을 거쳐 잎에 도착하는지 직접 따라가 보기 위해
sample = Xd[[2]]                                                        # 세 번째 꽃 (2차원 유지)
print("꽃 치수:", dict(zip(names_short, sample[0].round(2).tolist())))  # 측정값 4개
leaf = shallow_tree.apply(sample)[0]                                    # 도착 잎 번호
for node in shallow_tree.decision_path(sample).indices:                 # 지나간 상자들
    if node == leaf:                                                    # 잎은 질문이 없음
        continue                                                        # 건너뜀
    fi, th = t.feature[node], t.threshold[node]                         # 질문 피처 · 기준값
    go = "예, 왼쪽" if sample[0, fi] <= th else "아니오, 오른쪽"        # 답
    print(f"  {names_short[fi]} <= {th:.2f}?  관측 {sample[0, fi]:.2f}  →  {go}")  # 경로 한 줄
print("  예측:", target_names[shallow_tree.predict(sample)[0]],         # 다수 클래스
      "| predict_proba:", shallow_tree.predict_proba(sample)[0].round(4).tolist())  # 잎 비율

꽃 치수: {'sepal length': 6.8, 'sepal width': 2.8, 'petal length': 4.8, 'petal width': 1.4}
  petal length <= 2.45?  관측 4.80  →  아니오, 오른쪽
  petal width <= 1.65?  관측 1.40  →  예, 왼쪽
  예측: versicolor | predict_proba: [0.0, 0.9286, 0.0714]


**결과** · 예측 정답은 92.9% (39 ÷ 42)

- `petal length <= 2.45?  관측 4.80  →  아니오, 오른쪽` — 4.80 > 2.45
- `petal width <= 1.65?  관측 1.40  →  예, 왼쪽` — 1.40 ≤ 1.65 → 잎 3
- `예측: versicolor | predict_proba: [0.0, 0.9286, 0.0714]` — 39 ÷ 42 = 0.929 · 3 ÷ 42 = 0.071

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>apply(X)</code></td><td>도착 잎 번호</td><td>어느 잎인지</td></tr>
<tr><td><code>decision_path(X)</code></td><td>지나간 상자 번호</td><td>경로 설명</td></tr>
<tr><td><code>predict_proba(X)</code></td><td>잎의 클래스 비율</td><td>확신 정도 (잎이 작으면 거칠음)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 트리는 답을 외운 표가 아니라 순서 있는 질문 규칙
> - 잎이 작으면 비율이 거칠게 나옴 (예: 1 ÷ 1 = 1.0)

> ✅ **한 줄 정리** · 꽃 한 송이는 질문마다 예 / 아니오로 길을 따라 잎에 도착하고, 예측은 그 잎의 다수 클래스와 비율이다.

**딥러닝 코드에서는**

- 모델 설명이 필요하면 '이 입력이 왜 이 답인가'를 경로처럼 보여 줄 수 있어야 함

---
## 2. 좋은 질문 고르기

> **이 장의 흐름**
>
> - 두 후보 질문을 Count · Rate로 비교
> - 섞인 정도에 Gini라는 이름을 붙이고 양쪽 크기까지 감안해 계산

<img src="images/pfm6_split.png" alt="후보 A · B가 만든 두 그룹의 품종 개수" width="912">

**파이프라인 위치** · ④ 모델 — 분할 기준

### 2-1. `A` Count → Rate로 질문 비교 · `np.bincount` <sub>원본 셀 12 ~ 14</sub>

> 🎯 **A `petal length <= 2.45`와 B `sepal length <= 5.00` 중 어느 쪽이 한 그룹에 같은 종류를 더 많이 모으는가?**

💡 **쉽게 말하면**

- 스무고개 첫 질문은 '고양이인가?'보다 '**동물인가?**' → 세상이 크게 갈라짐.
- 트리도 같은 기준으로 고름.

**실행 흐름**

1. 질문마다 예 / 아니오 그룹
2. 그룹별 품종 개수
3. 가장 큰 비율(Rate)

**먼저 예측** · A와 B 중 어느 쪽이 좋은 질문일까? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · 질문 하나가 데이터를 얼마나 깨끗하게 나누는지 개수 · 비율로 보기 위해
def split_count_rate(fi, th):                                # 질문 하나 → 양쪽 품종 개수
    rows = []                                                # 예 · 아니오 두 줄
    for side, mask in [("예(왼쪽)", Xd[:, fi] <= th),        # 기준 이하
                       ("아니오(오른쪽)", Xd[:, fi] > th)]:  # 기준 초과
        c = np.bincount(yd[mask], minlength=3)               # 품종별 개수
        rows.append([side, int(mask.sum()), *c.tolist(), round(c.max() / mask.sum(), 4)])  # 한 줄
    return pd.DataFrame(rows, columns=["답", "전체", *target_names, "가장 큰 Rate"])  # 표


# 왜 · 좋은 질문과 나쁜 질문의 나눔 결과를 나란히 비교하기 위해
print("A: petal length <= 2.45")                         # 후보 A
print(split_count_rate(2, 2.45).to_string(index=False))  # 원본 셀 13
print("B: sepal length <= 5.00")                         # 후보 B
print(split_count_rate(0, 5.00).to_string(index=False))  # 원본 셀 13

A: petal length <= 2.45
       답  전체  setosa  versicolor  virginica  가장 큰 Rate
   예(왼쪽)  40      40           0          0        1.0
아니오(오른쪽)  80       0          40         40        0.5
B: sepal length <= 5.00
       답  전체  setosa  versicolor  virginica  가장 큰 Rate
   예(왼쪽)  25      23           1          1     0.9200
아니오(오른쪽)  95      17          39         39     0.4105


**결과** · 예측 정답은 A

- A · 왼쪽은 setosa 40/40 완전히 한 종류
  - `예(왼쪽)  40      40           0          0        1.0`
  - `아니오(오른쪽)  80       0          40         40        0.5`
- B · 왼쪽 92%는 깨끗해 보이지만 25개뿐 · 오른쪽 95개가 크게 섞임
  - `예(왼쪽)  25      23           1          1     0.9200`
  - `아니오(오른쪽)  95      17          39         39     0.4105`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>Count</td><td>그룹별 품종 개수</td><td>눈으로 먼저</td></tr>
<tr><td>Rate</td><td>그룹 안 가장 큰 비율</td><td>깨끗한 정도</td></tr>
<tr><td>크기</td><td>그룹이 몇 개를 담았나</td><td>작은 그룹의 깨끗함에 속지 않기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한쪽만 깨끗하면 안 됨 → 양쪽을 **크기까지 감안해** 비교 (2-2)

> ✅ **한 줄 정리** · 좋은 질문은 양쪽 그룹을 크기까지 감안했을 때 같은 종류끼리 더 많이 모으는 질문이다.

**딥러닝 코드에서는**

- 분류 헤드도 결국 클래스를 '잘 가르는' 경계를 찾는 일

### 2-2. `A` Gini 불순도 계산 · `1 - Σp²` · `가중 평균` <sub>원본 셀 15 ~ 17</sub>

> 🎯 **부모 Gini에서 A와 B는 각각 얼마나 낮추는가?**

💡 **쉽게 말하면**

- 섞인 정도 = **불순도** · 깨끗하면 0, 섞일수록 큼.
- Gini = 1 − Σ pᵢ² · 양쪽은 개수에 비례해 섞음.

**실행 흐름**

1. gini 함수 정의
2. 부모 · 양쪽 Gini
3. 개수 비율로 가중 평균

**먼저 예측** · Iris 루트(3품종 40 · 40 · 40)의 Gini는 0.5일까? — 답은 코드 실행 결과 아래에

In [6]:
# 왜 · 트리가 질문을 고르는 기준인 Gini 불순도를 직접 계산해 보기 위해
def gini(labels):                                       # 1 - Σ p²
    p = np.bincount(labels, minlength=3) / len(labels)  # 품종 비율
    return 1 - (p ** 2).sum()                           # 섞일수록 큼


# 왜 · 자식 그룹 크기를 반영한 가중 Gini로 질문의 좋고 나쁨을 숫자로 비교하기 위해
def weighted_child_gini(fi, th):                    # 양쪽을 개수 비율로
    left = Xd[:, fi] <= th                          # 예 그룹
    n, nl = len(yd), left.sum()                     # 전체 · 왼쪽 개수
    gl, gr = gini(yd[left]), gini(yd[~left])        # 양쪽 Gini
    return gl, gr, nl / n * gl + (n - nl) / n * gr  # 왼 · 오 · 가중 평균


# 왜 · 질문 전후로 불순도가 얼마나 줄었는지 확인하기 위해
print(f"부모 Gini: {gini(yd):.3f}")                                        # 3클래스 최대
for tag, fi, th in [("A", 2, 2.45), ("B", 0, 5.00)]:                       # 두 후보
    gl, gr, gw = weighted_child_gini(fi, th)                               # 계산
    print(f"{tag}: 왼쪽 {gl:.3f} · 오른쪽 {gr:.3f} · 크기 감안 {gw:.3f}")  # 원본 셀 17 표

부모 Gini: 0.667
A: 왼쪽 0.000 · 오른쪽 0.500 · 크기 감안 0.333
B: 왼쪽 0.150 · 오른쪽 0.631 · 크기 감안 0.531


**결과** · 예측 정답은 아님 — 0.667 (3클래스)

- `부모 Gini: 0.667` — 1 − 3 × (1/3)² = 0.667 · 클래스 C개면 최대 1 − 1/C
- `A: 왼쪽 0.000 · 오른쪽 0.500 · 크기 감안 0.333` — 왼쪽 0과 오른쪽 0.5를 40 : 80 비율로 섞어 0.333
- `B: 왼쪽 0.150 · 오른쪽 0.631 · 크기 감안 0.531` — 부모에서 덜 낮춤 → A가 이김

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>2클래스 Gini</td><td>한쪽만 0 · 0.8 / 0.2 → 0.32 · 반반 0.5</td><td>최대 0.5</td></tr>
<tr><td>3클래스 Gini</td><td>최대 1 − 1/3 = 0.667</td><td>Iris 루트</td></tr>
<tr><td>트리가 하는 일</td><td>가능한 질문을 전부 시험해 가중 Gini를 가장 많이 낮추는 하나</td><td>분할마다 반복</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 최대 0.5는 **2클래스일 때만** · 3클래스 Iris 루트는 0.667

> ✅ **한 줄 정리** · 트리는 부모 Gini를 크기 감안한 자식 Gini로 가장 많이 낮추는 질문을 고르며, A(0.667 → 0.333)가 B(0.531)보다 낫다.

**딥러닝 코드에서는**

- 교차 엔트로피도 '섞임'을 재는 같은 계열 (엔트로피 → 6-2)

---
## 3. 질문 횟수 제한

> **이 장의 흐름**
>
> - 깊이 1 → 끝까지 늘리며 train과 CV(처음 보는 문제) 정답률 비교
> - 그다음 sepal 두 피처로 깊이 제한 없음 vs 3의 경계 모양 (Support E)

<img src="images/pfm6_depth.png" alt="깊이별 train vs CV 정답률" width="605">

**파이프라인 위치** · ④ 모델 — 복잡도 · 과적합

### 3-1. `A` 깊이별 train vs CV · `max_depth` · `cross_val_predict` <sub>원본 셀 18 ~ 22</sub>

> 🎯 **깊이를 1 → 2 → 3 → 4 → 5 → 8 → 제한 없음으로 늘리면 train과 CV는 함께 오르는가?**

💡 **쉽게 말하면**

- `max_depth=3` = 한 경로에서 **질문을 최대 3번**.
- 질문을 더 하면 섞인 3개도 걸러내지만, 처음 보는 문제까지 잘 풀지는 않음.

**실행 흐름**

1. 깊이 후보 7개
2. CV 정답 개수 · 평균 · 표준편차
3. train 정답률 · 잎 수

**먼저 예측** · CV 정답 개수도 계속 오를까? — 답은 코드 실행 결과 아래에

In [7]:
# 왜 · 모든 깊이를 같은 CV 계약으로 비교해 공정하게 고르기 위해
def cv_report(model, Xv, yv):                         # 원본 도우미: 맞힌 개수 · 평균 · 표준편차
    oof = cross_val_predict(model, Xv, yv, cv=CV)     # fold마다 처음 보는 예측
    s = cross_val_score(model, Xv, yv, cv=CV)         # fold별 정답률
    return int((oof == yv).sum()), s.mean(), s.std()  # 세 값


rows = []                                                                   # 깊이마다
for depth in [1, 2, 3, 4, 5, 8, None]:                                      # 원본 셀 19의 후보
    m = DT(max_depth=depth, random_state=RS)                                # 질문 횟수 제한
    c, mean, std = cv_report(m, Xd, yd)                                     # 처음 보는 문제
    m.fit(Xd, yd)                                                           # 외운 문제용 학습
    tr = accuracy_score(yd, m.predict(Xd))                                  # train 정답률
    rows.append([str(depth), tr, c, mean, std, m.get_n_leaves()])           # 한 줄
cols = ["max_depth", "train", "cv_correct", "cv_mean", "cv_std", "leaves"]  # 원본 열 이름
depth_df = pd.DataFrame(rows, columns=cols)                                 # 깊이별 표
print(depth_df.round(4).to_string(index=False))                             # 원본 표

max_depth  train  cv_correct  cv_mean  cv_std  leaves
        1 0.6667          80   0.6667  0.0000       2
        2 0.9667         113   0.9417  0.0204       3
        3 0.9833         115   0.9583  0.0264       5
        4 0.9917         113   0.9417  0.0204       7
        5 1.0000         114   0.9500  0.0167       8
        8 1.0000         114   0.9500  0.0167       8
     None 1.0000         114   0.9500  0.0167       8


**결과** · 예측 정답은 아님 — 80 → 113 → 115 → 113 → 114 …

- `1  0.6667          80   0.6667  0.0000       2` — 질문 1번 → 세 종류를 못 나눔 (과소적합)
- `3  0.9833         115   0.9583  0.0264       5` — CV 정답 115개로 가장 높음
- `None  1.0000         114   0.9500  0.0167       8` — train은 1.0 천장 · CV는 안 따라옴 (과적합)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>과소적합 (깊이 1)</td><td>train · CV 둘 다 낮음</td><td>질문을 늘림</td></tr>
<tr><td>적당 (깊이 2 ~ 3)</td><td>CV 최고 근처</td><td>고를 후보</td></tr>
<tr><td>과적합 (깊이 5 이상)</td><td>train 1.0 · CV 정체</td><td>깊이 · 잎 크기를 제한</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 깊이 5 · 8 · None은 같은 트리(잎 8개) → 데이터가 작으면 더 깊어질 곳이 없음
> - CV 1건 = 0.83%p → 113 vs 115는 2건 차이

> ✅ **한 줄 정리** · 깊이를 늘리면 train은 1.0까지 쉽게 오르지만 처음 보는 문제 점수는 따라오지 않으며, 이 어긋남이 과적합이다.

**딥러닝 코드에서는**

- 층 · epoch을 늘리면 train loss는 계속 내려가도 val은 멈춤 — 같은 그림

### 3-2. `B` 경계 모양 · sepal 두 개 · `predict(grid)` · `get_n_leaves` <sub>원본 셀 55 ~ 59 (Support E)</sub>

> 🎯 **sepal 두 피처만 쓰면 깊이 제한 없음과 깊이 3 중 처음 보는 꽃을 더 맞히는 쪽은?**

💡 **쉽게 말하면**

- 경계를 그리려면 피처가 2개여야 함 → 트리가 안 쓴 **sepal**로 하면 깊이 차이가 드러남.
- 질문은 `피처 <= 값` 하나뿐 → 경계는 **축에 수직인 계단**.

<img src="images/pfm6_boundary.png" alt="sepal 두 피처 · 깊이 제한 없음 vs 3의 결정 경계" width="913">

**실행 흐름**

1. 개발 120을 84 / 36으로 (봉투는 안 엶)
2. 깊이 제한 없음 · 3 두 트리
3. 잎 · train · unseen 정답 수

**먼저 예측** · train을 잘 맞히는 쪽이 unseen도 잘 맞힐까? — 답은 코드 실행 결과 아래에

In [8]:
# 왜 · 특성 두 개만 써서 깊은 트리가 학습 데이터를 외우는 과적합을 눈으로 보기 위해
X2 = Xd[:, [0, 1]]                                                     # sepal length · width만
X2_tr, X2_un, y2_tr, y2_un = train_test_split(                         # 84 / 36 (봉투는 그대로)
    X2, yd, test_size=0.30, stratify=yd, random_state=RS)
dt_free = DT(random_state=RS).fit(X2_tr, y2_tr)                        # 제한 없음
dt_3 = DT(max_depth=3, random_state=RS).fit(X2_tr, y2_tr)              # 깊이 3
for label, m in [("max_depth=None", dt_free), ("max_depth=3", dt_3)]:  # 두 트리
    tr_ok, un_ok = (m.predict(X2_tr) == y2_tr).sum(), (m.predict(X2_un) == y2_un).sum()  # 정답 수
    # 원본 셀 56
    print(f"[{label}] 잎 {m.get_n_leaves():2d} | train {tr_ok}/84 | unseen {un_ok}/36")

[max_depth=None] 잎 31 | train 82/84 | unseen 26/36
[max_depth=3] 잎  7 | train 69/84 | unseen 31/36


**결과** · 예측 정답은 아님 — 완전히 뒤집힘

- `[max_depth=None] 잎 31 | train 82/84 | unseen 26/36` — train 97.6% · unseen 72.2%
- `[max_depth=3] 잎  7 | train 69/84 | unseen 31/36` — train 82.1% · unseen 86.1%

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>로지스틱 회귀</td><td>사선 경계를 직선 하나로</td><td>경계가 비스듬할 때 강함</td></tr>
<tr><td>결정 트리</td><td>계단을 이어 붙여 사선을 흉내</td><td>깊이를 많이 씀 → 과적합 위험</td></tr>
<tr><td>깊이 제한</td><td>계단 수를 줄임</td><td>처음 보는 점을 덜 놓침</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - train에서 97.6% − 82.1% = 15.5%p 앞섰지만 unseen에서는 86.1% − 72.2% = 13.9%p 뒤집힘
> - sepal 두 개는 5-2에서 중요도 0.000인 피처 → 그래도 31/36을 맞힘

> ✅ **한 줄 정리** · 트리는 축에 수직인 계단으로만 경계를 그려서, 깊이 제한이 없으면 train을 감싸느라 처음 보는 점을 놓친다.

**딥러닝 코드에서는**

- 신경망 경계는 매끈한 곡선 · 트리 앙상블은 계단의 합

---
## 4. 가지치기와 후보 비교

> **이 장의 흐름**
>
> - min_samples_split · leaf로 잎 수 줄이기 → GridSearchCV로 36조합 비교 · 공동 1위 확인 → 선택 종료
> - 그다음 CV만 바꿔 같은 탐색을 다시 (Support D)

<img src="images/pfm6_contract.png" alt="계약 = 후보 · 지표 · 분할 · 하나만 바꿔도 선택이 바뀜" width="763">

**파이프라인 위치** · ⑤ 교차검증 · 튜닝 — 선택 종료

### 4-1. `B` min_samples_split · leaf · `min_samples_split` · `min_samples_leaf` <sub>원본 셀 23 ~ 26</sub>

> 🎯 **깊이 4 트리에 split=8 · leaf=4를 더하면 잎 수와 CV는 어떻게 되는가?**

💡 **쉽게 말하면**

- `split`은 **나누기 전** 최소 개수 → 그냥 멈춤.
- `leaf`는 **나눠 본 뒤** 자식 최소 개수 → 다른 분할을 찾음.

**실행 흐름**

1. 깊이 4만
2. 깊이 4 + split 8 + leaf 4
3. CV · 잎 수 비교

**먼저 예측** · 잎이 줄면 CV 정답 수(113)도 오를까? — 답은 코드 실행 결과 아래에

In [9]:
# 왜 · 가지치기 조건이 복잡도를 줄이는지와 점수가 어떻게 되는지 함께 확인하기 위해
pruned = DT(max_depth=4, min_samples_split=8, min_samples_leaf=4, random_state=RS)  # 가지치기 추가
# 두 후보
cands = {"depth=4 only": DT(max_depth=4, random_state=RS), "depth=4 + split/leaf": pruned}
for label, m in cands.items():                                                      # 두 후보
    c, mean, std = cv_report(m, Xd, yd)                                             # 같은 CV 계약
    leaves_n = m.fit(Xd, yd).get_n_leaves()                                         # 잎 수
    # 원본 셀 24
    print(f"{label:<22s} cv {c}/120 · mean {mean:.4f} · std {std:.4f} · 잎 {leaves_n}")

depth=4 only           cv 113/120 · mean 0.9417 · std 0.0204 · 잎 7


depth=4 + split/leaf   cv 113/120 · mean 0.9417 · std 0.0204 · 잎 5

**결과** · 예측 정답은 아님 — 잎 7 → 5 · CV 113 → 113

- `depth=4 only           cv 113/120 · mean 0.9417 · std 0.0204 · 잎 7` — 기준
- `depth=4 + split/leaf   cv 113/120 · mean 0.9417 · std 0.0204 · 잎 5` — 트리는 단순해졌지만 맞힌 개수는 그대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>max_depth</code></td><td>최대 몇 번 질문</td><td>가장 먼저 조절</td></tr>
<tr><td><code>min_samples_split</code></td><td>나누기 전 최소 개수 → 그냥 멈춤</td><td>작은 노드에서 멈추게</td></tr>
<tr><td><code>min_samples_leaf</code></td><td>자식이 작아지는 분할은 후보에서 제외 → 다른 분할</td><td>잎이 너무 작지 않게</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 가지치기는 성능을 올려 주는 버튼이 아니라 **복잡도를 제한하는 도구**
> - 단순해진 것과 새 데이터를 더 잘 맞히는 것은 다른 이야기

> ✅ **한 줄 정리** · 가지치기는 트리를 단순하게 만들지만 성능 보장은 아니며, 효과는 같은 CV로 확인한다.

**딥러닝 코드에서는**

- dropout · weight decay도 복잡도 제한 · 효과는 val로 확인

### 4-2. `A` GridSearchCV · 공동 1위 · 선택 종료 · `GridSearchCV` · `rank_test_score` · `freeze_selection` <sub>원본 셀 26 ~ 29</sub>

> 🎯 **36조합을 같은 계약으로 비교하면 1위는 하나인가?**

💡 **쉽게 말하면**

- **계약** = 비교 전에 정하는 세 가지 · ① 후보 ② 점수 ③ 데이터 나누는 법.
- 결과는 '세상에서 제일 좋은 트리'가 아니라 **이 계약 안에서 뽑힌 후보**.

**실행 흐름**

1. 깊이 4 × split 3 × leaf 3 = 36조합
2. 5-Fold로 180번 학습 + refit 1번
3. 공동 1위 수 · 선택 종료

**먼저 예측** · 1위는 몇 개일까? — 답은 코드 실행 결과 아래에

In [10]:
# 왜 · 여러 가지치기 조합을 같은 CV로 자동 탐색하기 위해
param_grid = {"max_depth": [2, 3, 4, None], "min_samples_split": [2, 4, 8],  # 원본 셀 27
              "min_samples_leaf": [1, 2, 4]}                                 # 4 × 3 × 3 = 36
gs = GridSearchCV(DT(random_state=RS), param_grid,                           # 같은 계약
                  cv=CV, scoring="accuracy").fit(Xd, yd)
n_tied = int((pd.DataFrame(gs.cv_results_)["rank_test_score"] == 1).sum())   # 공동 1위 수
print("선택된 후보:", gs.best_params_)                                       # 동점이면 순서상 먼저
print(f"CV mean accuracy: {gs.best_score_:.4f} · 36개 조합 중 공동 1위: {n_tied}개")  # 원본 셀 27
best_dt = gs.best_estimator_  # 개발 120개로 다시 학습된 트리
FINAL_HOLDOUT.freeze_selection()  # 이제 선택을 되돌리는 코드는 없음
print("SELECTION FROZEN · open count =", FINAL_HOLDOUT.open_count)           # 아직 0

선택된 후보: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
CV mean accuracy: 0.9583 · 36개 조합 중 공동 1위: 16개
SELECTION FROZEN · open count = 0


**결과** · 예측 정답은 16개 공동 1위

- `선택된 후보: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}` — 공동 1위 중 순서상 먼저
- `CV mean accuracy: 0.9583 · 36개 조합 중 공동 1위: 16개` — best_params_는 유일한 정답이 아님
- `SELECTION FROZEN · open count = 0` — 선택 종료 · 봉투는 아직 안 열림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>① 후보</td><td>param_grid 36조합</td><td>범위를 사람이 정함</td></tr>
<tr><td>② 점수</td><td>accuracy</td><td>업무에 맞는 지표로</td></tr>
<tr><td>③ 분할</td><td>StratifiedKFold(5, shuffle=True, 42)</td><td>바뀌면 답도 바뀜 (4-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - sklearn은 동점일 때 순서상 먼저 나온 후보를 돌려줌 → best_params_ = 동점 후보 중 하나

> ✅ **한 줄 정리** · GridSearchCV 결과는 계약(후보 · 점수 · 분할) 안에서 뽑힌 동점 후보 중 하나이며, 선택을 끝낸 뒤에만 봉투를 연다.

**딥러닝 코드에서는**

- 학습률 탐색 결과도 '이 seed · 이 val 분할에서'라는 계약과 함께 공유

### 4-3. `B` 계약이 바뀌면 선택도 바뀜 · `cv=5` · `cv=CV` <sub>원본 셀 52 ~ 54 (Support D)</sub>

> 🎯 **같은 데이터 · 후보 · 지표에서 fold 나누는 방식만 바꾸면 선택이 바뀌는가?**

💡 **쉽게 말하면**

- 이전 버전 강의는 `max_depth=4`를 골랐고 이번은 `3`.
- 다른 것은 **fold를 나누는 방식 하나**.

**실행 흐름**

1. cv=5 (섞지 않음)
2. cv=StratifiedKFold(shuffle=True)
3. 고른 깊이 · 점수 · 공동 1위

**먼저 예측** · cv만 바꿔도 max_depth가 바뀔까? — 답은 코드 실행 결과 아래에

In [11]:
# 왜 · CV 설정이 달라지면 점수도 달라진다는 것을 보여 같은 계약을 지켜야 하는 이유를 확인하기 위해
contracts = [("이전: cv=5 (shuffle 없음)", 5),                               # 섞지 않은 5등분
             ("이번: StratifiedKFold(5, shuffle=True, 42)", CV)]             # 오늘의 CV
for label, cv_obj in contracts:                                              # 분할 방식만 다름
    s = GridSearchCV(DT(random_state=RS), param_grid,                        # 후보 · 지표 같음
                     cv=cv_obj, scoring="accuracy").fit(Xd, yd)
    tied = int((pd.DataFrame(s.cv_results_)["rank_test_score"] == 1).sum())  # 공동 1위
    d_best = s.best_params_["max_depth"]                                     # 고른 깊이
    # 원본 셀 53
    print(f"{label} → max_depth = {d_best}, CV = {s.best_score_:.4f}, 공동 1위 = {tied}개")

이전: cv=5 (shuffle 없음) → max_depth = 4, CV = 0.9417, 공동 1위 = 2개


이번: StratifiedKFold(5, shuffle=True, 42) → max_depth = 3, CV = 0.9583, 공동 1위 = 16개


**결과** · 예측 정답은 바뀜 — 4 → 3

- `이전: cv=5 (shuffle 없음) → max_depth = 4, CV = 0.9417, 공동 1위 = 2개` — 섞지 않은 등분
- `이번: StratifiedKFold(5, shuffle=True, 42) → max_depth = 3, CV = 0.9583, 공동 1위 = 16개` — 섞은 등분 · 선택이 3으로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>숫자만 공유</td><td>'max_depth=3이 최고'</td><td>쓰지 않기</td></tr>
<tr><td>계약과 함께 공유</td><td>'이 후보 · 이 지표 · 이 CV에서 3'</td><td><strong>올바른 공유</strong></td></tr>
<tr><td>선택의 정답</td><td>없음 · 계약 안에서의 선택</td><td>결과에 조건을 붙임</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 비교도 봉투를 열지 않음 → 개발 120개 안에서만

> ✅ **한 줄 정리** · 같은 데이터 · 후보 · 지표라도 fold 구성만 바뀌면 선택이 바뀌므로, 결과는 계약과 함께 공유한다.

**딥러닝 코드에서는**

- seed · val 분할을 바꾸면 고른 하이퍼파라미터가 바뀌는 것은 흔한 일

---
## 5. 봉투 열기와 해석

> **이 장의 흐름**
>
> - 선택 종료 뒤 봉투를 한 번만 열어 최종 점수와 혼동행렬
> - 그다음 이미 정해진 트리의 중요도를 범위를 지켜 해석

<img src="images/pfm6_importance.png" alt="선택된 트리의 feature_importances_" width="608">

**파이프라인 위치** · ⑥ 최종 평가 1회 → 해석

### 5-1. `A` 봉투를 한 번만 열기 · `open_once` · `confusion_matrix` <sub>원본 셀 30 ~ 34</sub>

> 🎯 **봉인한 30개 중 몇 개를 맞히고, CV보다 높게 나오면 무슨 뜻인가?**

💡 **쉽게 말하면**

- 결과가 마음에 안 들어도 **그걸 보고 다시 고르지 않음**.
- 30개 평가에서 1건 = 3.3%p → 작은 차이는 흔들림.

**실행 흐름**

1. 봉투 열기 (한 번)
2. 예측 · 정답 수
3. 혼동행렬 · 연 횟수

**먼저 예측** · 30개 중 몇 개를 맞힐까? — 답은 코드 실행 결과 아래에

In [12]:
# 왜 · 선택이 끝난 모델로 최종 데이터를 한 번만 채점하기 위해
X_final, y_final = FINAL_HOLDOUT.open_once()                            # 선택 종료 뒤 단 한 번
y_pred_final = best_dt.predict(X_final)                                 # 고른 트리 그대로
final_correct = int((y_pred_final == y_final).sum())                    # 맞힌 개수
print(f"FINAL_HOLDOUT: {final_correct}/30 = {final_correct / 30:.1%}")  # 원본 셀 31
print(f"참고: 30개 평가에서 1건 = {1 / 30:.1%}p · development CV: {gs.best_score_:.1%}")  # 비교
# 어디서 틀렸나
print("혼동행렬 (행 = 실제, 열 = 예측):", confusion_matrix(y_final, y_pred_final).tolist())
print(f"open count = {FINAL_HOLDOUT.open_count}")                       # 1
del X_final, y_final, y_pred_final                                      # 다시 못 쓰게 지움

FINAL_HOLDOUT: 29/30 = 96.7%
참고: 30개 평가에서 1건 = 3.3%p · development CV: 95.8%
혼동행렬 (행 = 실제, 열 = 예측): [[10, 0, 0], [0, 9, 1], [0, 0, 10]]
open count = 1


**결과** · 예측 정답은 29/30 (96.7%)

- `FINAL_HOLDOUT: 29/30 = 96.7%` — 1건 틀림
- `참고: 30개 평가에서 1건 = 3.3%p · development CV: 95.8%` — CV보다 높지만 1건 차이 수준
- `혼동행렬 (행 = 실제, 열 = 예측): [[10, 0, 0], [0, 9, 1], [0, 0, 10]]` — versicolor 1건을 virginica로 · setosa는 0건
- `open count = 1` — 봉투는 한 번만 열림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>120개에서 1건</td><td>0.83%p</td><td>CV 차이 읽기</td></tr>
<tr><td>30개에서 1건</td><td>3.3%p</td><td>봉투 점수 읽기</td></tr>
<tr><td>다시 고르기</td><td>새 평가 데이터가 필요</td><td>같은 봉투로는 안 됨</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 최종 96.7%가 CV 95.8%보다 높다고 '봉인하면 좋아진다'로 읽으면 안 됨 → 30개에서 흔한 흔들림
> - 오분류는 versicolor · virginica가 겹치는 경계(1장 잎 [0, 39, 3])에서 남

> ✅ **한 줄 정리** · 선택을 끝낸 뒤 봉투를 한 번만 열어 보고하며, 작은 평가에서는 1건이 3.3%p라 점수 차이를 크기까지 감안해 읽는다.

**딥러닝 코드에서는**

- test loader를 함수로 감싸 한 번만 돌게 하는 것도 같은 생각

### 5-2. `A` 중요도 0 ≠ 쓸모없음 · `feature_importances_` <sub>원본 셀 35 ~ 38</sub>

> 🎯 **선택된 트리는 피처 4개를 다 썼는가? 중요도 0은 무슨 뜻인가?**

💡 **쉽게 말하면**

- 중요도 = **이 트리**가 분기에서 줄인 불순도의 비중 · 다 더하면 1.
- 0 = 이 트리의 분기에 안 쓰였다 ≠ 정보가 없다.

**실행 흐름**

1. feature_importances_
2. 합 확인
3. sepal만으로도 맞힌 3-2와 비교

**먼저 예측** · sepal 두 피처의 중요도는? — 답은 코드 실행 결과 아래에

In [13]:
# 왜 · 모델이 어떤 특성을 많이 썼는지 확인해 결과를 설명하기 위해
imp = pd.Series(best_dt.feature_importances_, index=names_short)  # 피처별 중요도
imp = imp.sort_values(ascending=False)                            # 큰 순서
print(imp.round(6).to_string())                                   # 원본 셀 36
print("합계:", round(imp.sum(), 6))                               # 정규화돼 1
# 분기에 쓴 것
print("이 트리가 쓴 피처:", [names_short[i] for i in sorted(set(best_dt.tree_.feature)) if i >= 0])

petal length    0.579077
petal width     0.420923
sepal width     0.000000
sepal length    0.000000
합계: 1.0
이 트리가 쓴 피처: ['petal length', 'petal width']


**결과** · 예측 정답은 0.000 · 0.000

- `petal length    0.579077` — 가장 많이 쓴 단서
- `petal width     0.420923` — 두 번째
- `sepal length    0.000000` — 이 트리의 분기에 안 쓰임
- `이 트리가 쓴 피처: ['petal length', 'petal width']` — petal 두 개가 먼저 일을 끝냄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>중요도 0</td><td>이 트리에서 안 쓰임</td><td>'쓸모없다'가 아님 (sepal만으로 31/36)</td></tr>
<tr><td>중요도 큼</td><td>이 트리가 많이 쓴 단서</td><td>원인이 아님 (우산 · 비)</td></tr>
<tr><td>비슷한 피처</td><td>중요도를 나눠 가짐</td><td>하나만 남기면 몰아감</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 데이터가 바뀌면 중요도도 바뀜 · seed만 바꿔도 바뀜 (6-1)
> - **사용되지 않은 피처 ≠ 중요하지 않은 피처** · 중요도는 인과관계가 아님

> ✅ **한 줄 정리** · feature_importances_는 이 트리가 쓴 단서의 비중일 뿐이라, 0은 '쓸모없다'가 아니라 '이번엔 쓸 자리가 없었다'는 뜻이다.

**딥러닝 코드에서는**

- Grad-CAM · attention 지도도 '모델이 본 곳'이지 원인이 아님

---
## 6. 흔들림과 보조 실험

> **이 장의 흐름**
>
> - 같은 120개 · seed 0 ~ 29로 30그루의 루트 질문 세기 (Act 7)
> - 그다음 Support A · B · C · Gini vs Entropy · NaN 처리 · class_weight

<img src="images/pfm6_roots.png" alt="seed 30개 · 서로 다른 루트 질문" width="662">

**파이프라인 위치** · ④ 모델 — 안정성 · 보조 설정

### 6-1. `A` seed만 바꾼 30그루 · `random_state=s` · `groupby().size()` <sub>원본 셀 39 ~ 42</sub>

> 🎯 **같은 120개에서 random_state만 0 ~ 29로 바꾸면 루트 질문은 몇 가지가 나오는가?**

💡 **쉽게 말하면**

- 트리는 if-else인데 왜 랜덤? → **동점** 때문.
- 불순도를 똑같이 줄이는 질문이 둘 이상이면 무작위로 하나를 고름.

**실행 흐름**

1. seed 30개로 트리 30그루
2. 루트 질문 문자열
3. 질문별 횟수

**먼저 예측** · 루트 질문은 몇 가지? — 답은 코드 실행 결과 아래에

In [14]:
# 왜 · seed가 바뀌면 첫 질문도 바뀔 수 있다는 트리의 불안정성을 보기 위해
def describe_root(m):                                                # 루트 질문을 글자로
    tt = m.tree_                                                     # 내부 구조
    return f"{names_short[tt.feature[0]]} <= {tt.threshold[0]:.2f}"  # 첫 질문


roots = [describe_root(DT(random_state=s).fit(Xd, yd))   # 같은 데이터
         for s in range(30)]                             # seed만 0 ~ 29
print(pd.Series(roots).value_counts().to_string())       # 원본 셀 40
print("데이터는 단 한 개도 바뀌지 않았음 (같은 120개)")  # 확인

petal width <= 0.80     22
petal length <= 2.45     8
데이터는 단 한 개도 바뀌지 않았음 (같은 120개)


**결과** · 예측 정답은 2가지 (22번 · 8번)

- `petal width <= 0.80     22` — 다수파
- `petal length <= 2.45     8` — 1장에서 본 질문은 30번 중 8번뿐
- `데이터는 단 한 개도 바뀌지 않았음 (같은 120개)` — 두 질문 모두 setosa 40개를 똑같이 완벽히 떼어 냄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>읽을 수 있음</td><td>딱 잘라 나눔 → 그림 한 장</td><td>트리의 장점</td></tr>
<tr><td>안정적임</td><td>같은 데이터를 설명하는 규칙이 하나가 아님</td><td>트리의 약점</td></tr>
<tr><td>보고서 실무</td><td>트리 그림 · 중요도에 <code>random_state</code>를 함께</td><td>재현 가능하게</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 성능이 나빠진 것은 아님 → 두 트리 다 잎 8개 · train 120/120
> - 트리가 읽히는 이유(딱 잘라 나눔)가 흔들리는 이유이기도 함

> ✅ **한 줄 정리** · 데이터를 하나도 안 바꿔도 seed만 바꾸면 동점 질문 중 다른 것을 골라 규칙이 바뀐다.

**딥러닝 코드에서는**

- seed마다 다른 가중치에 도착해도 성능은 비슷한 것과 같은 현상

### 6-2. `B` Gini vs Entropy · `criterion="entropy"` <sub>원본 셀 46 ~ 48 (Support A)</sub>

> 🎯 **분할 기준을 Gini에서 Entropy로 바꾸면 CV는 달라지는가?**

💡 **쉽게 말하면**

- Entropy = 불확실성의 양을 재는 다른 방법.
- 2클래스 기준 Gini 0 ~ 0.5, Entropy 0 ~ 1 · 원리는 같음.

**실행 흐름**

1. 깊이 3 · 기준만 다른 트리 두 개
2. 같은 CV로 정답 수

**먼저 예측** · 두 기준의 차이는 몇 건일까? — 답은 코드 실행 결과 아래에

In [15]:
# 왜 · 분할 기준을 바꿔도 결과가 크게 다르지 않은지 비교하기 위해
for name, crit in [("Gini", "gini"), ("Entropy", "entropy")]:  # 기준 두 개
    m = DT(max_depth=3, criterion=crit, random_state=RS)       # 깊이는 같음
    c, mean, _ = cv_report(m, Xd, yd)                          # 같은 CV
    print(f"{name:8s}: {c}/120 = {mean:.1%}")                  # 원본 셀 47

Gini    : 115/120 = 95.8%
Entropy : 114/120 = 95.0%


**결과** · 예측 정답은 1건 (115 vs 114)

- `Gini    : 115/120 = 95.8%` — 기본값
- `Entropy : 114/120 = 95.0%` — 1건 차이 → 우열 선언 안 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>Gini</td><td>1 − Σp² · 계산이 가벼움</td><td><strong>기본값</strong></td></tr>
<tr><td>Entropy</td><td>−Σp log p · 불확실성의 양</td><td>정보 이득으로 설명할 때</td></tr>
<tr><td>1건 차이</td><td>흔들림 수준</td><td>기준 선택으로 성능을 기대하지 않기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 폭으로 criterion의 우열을 선언하지 않음

> ✅ **한 줄 정리** · Gini와 Entropy는 섞임을 재는 두 방법일 뿐이라 결과 차이가 작으며, 기본값 Gini를 쓰면 된다.

**딥러닝 코드에서는**

- 교차 엔트로피 손실 = 같은 엔트로피 계열

### 6-3. `B` 결측치(NaN) 그대로 학습 · `make_classification` · `SimpleImputer` <sub>원본 셀 48 ~ 50 (Support B)</sub>

> 🎯 **sklearn 트리는 NaN을 직접 처리하는가? 평균으로 채운 것과 비교하면?**

💡 **쉽게 말하면**

- sklearn **1.3부터** 기본 설정 트리는 NaN을 직접 처리.
- 분기마다 결측을 왼쪽 · 오른쪽으로 보내 보고 더 나은 쪽을 기록.

**실행 흐름**

1. 가짜 데이터 300 × 4 · 10% 결측
2. NaN 그대로 트리
3. 평균으로 채운 트리

**먼저 예측** · 두 방식의 정답 수 차이는 클까? — 답은 코드 실행 결과 아래에

In [16]:
# 왜 · 결측이 있는 데이터에서 트리를 쓰는 방법을 시험할 가짜 데이터를 만들기 위해
from sklearn.datasets import make_classification  # 가짜 분류 데이터
from sklearn.impute import SimpleImputer          # 평균으로 채우기

Xm, ym = make_classification(n_samples=300, n_features=4, n_informative=3,       # 원본 셀 49
                             n_redundant=0, random_state=RS)
Xm[np.random.RandomState(RANDOM_STATE).rand(*Xm.shape) < 0.10] = np.nan          # 10% 결측
Xtr, Xte, ytr, yte = train_test_split(                                           # 7 : 3
    Xm, ym, test_size=0.3, stratify=ym, random_state=RS)
native = DT(max_depth=3, random_state=RS).fit(Xtr, ytr)                          # NaN 그대로
imp = SimpleImputer(strategy="mean")                                             # train 평균으로
Xtr_f = imp.fit_transform(Xtr)  # train 평균으로 채운 학습 데이터
filled = DT(max_depth=3, random_state=RS).fit(Xtr_f, ytr)                        # 채운 데이터로
print(f"결측 {int(np.isnan(Xm).sum())}개 / 전체 {Xm.size}개")                    # 결측 수
print(f"NaN 그대로 학습     : {(native.predict(Xte) == yte).sum()}/{len(yte)}")  # 직접 처리
# 채움
print(f"SimpleImputer(mean) : {(filled.predict(imp.transform(Xte)) == yte).sum()}/{len(yte)}")

결측 128개 / 전체 1200개
NaN 그대로 학습     : 76/90
SimpleImputer(mean) : 76/90


**결과** · 예측 정답은 작음 — 이 환경에서는 같음 (원본은 1건 차이)

- `결측 128개 / 전체 1200개` — 약 10%
- `NaN 그대로 학습     : 76/90` — 트리의 결측 경로 사용 · 원본(sklearn 1.6)은 75/90
- `SimpleImputer(mean) : 76/90` — 이 환경에서는 같음 → 어느 쪽이 항상 낫다고 말할 수 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>트리 (sklearn 1.3+)</td><td>NaN 직접 처리 · 분기마다 결측 방향 기록</td><td>결측이 의미 있을 때</td></tr>
<tr><td>로지스틱 · KNN</td><td>NaN을 그대로 못 받음</td><td>채우기 등 전처리 필요</td></tr>
<tr><td>채우기 (Imputer)</td><td>train 통계로 채움 (Pipeline 안에)</td><td>모델 공통 기본</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이전 자료의 '트리는 NaN을 못 다룬다' · '거리를 계산하지 않아서 처리한다'는 모두 정정 대상 (원본 설명)
> - 지원 범위는 버전 · estimator · 설정마다 다름 → 쓰는 환경의 문서 확인 · 중요한 건 **왜 비었는가**

> ✅ **한 줄 정리** · sklearn 1.3 이상의 기본 트리는 NaN을 분기별 결측 경로로 직접 처리하지만, 채우기와의 우열은 데이터마다 다르다.

**딥러닝 코드에서는**

- 결측 마스크를 입력 채널로 함께 넣는 방법도 있음

### 6-4. `B` 불균형 · class_weight · `class_weight="balanced"` · `average_precision_score` <sub>원본 셀 50 ~ 52 (Support C)</sub>

> 🎯 **정상 900 · 이상 100에서 트리에 class_weight='balanced'를 걸면 재현율 · 정밀도 · AP는?**

💡 **쉽게 말하면**

- '전부 정상'이라 해도 정확도 90% → 소수 클래스 오분류 벌점을 키움.
- 분류력을 키운다기보다 **판단 기준선을 소수 쪽으로** 옮김.

**실행 흐름**

1. 90 : 10 가짜 데이터
2. 가중치 없음 · balanced
3. 재현율 · 정밀도 · AP

**먼저 예측** · balanced는 AP도 크게 올릴까? — 답은 코드 실행 결과 아래에

In [17]:
# 왜 · 불균형 데이터에서는 정확도보다 AP가 더 정직하다는 것을 보이기 위해
from sklearn.metrics import average_precision_score  # PR 곡선 요약

Xi, yi = make_classification(n_samples=1000, n_features=4,  # 정상 90 : 이상 10
                             n_informative=3, n_redundant=0, weights=[0.9, 0.1], random_state=RS)
Xtr2, Xte2, ytr2, yte2 = train_test_split(                                            # 7 : 3
    Xi, yi, test_size=0.3, stratify=yi, random_state=RS)
for label, kw in [("no weight", {}), ("balanced", {"class_weight": "balanced"})]:     # 두 설정
    m = DT(max_depth=3, random_state=RS, **kw).fit(Xtr2, ytr2)                        # 깊이 3
    pred, proba = m.predict(Xte2), m.predict_proba(Xte2)[:, 1]                        # 예측 · 확률
    # 칸 수
    tp, n_pos, n_pp = int(((pred == 1) & (yte2 == 1)).sum()), int(yte2.sum()), int(pred.sum())
    ap = average_precision_score(yte2, proba)                                         # 순위 요약
    rec, prec = f"{tp}/{n_pos} = {tp / n_pos:.1%}", f"{tp}/{n_pp} = {tp / n_pp:.1%}"  # 분수 · 비율
    print(f"[{label:9s}] recall {rec} | precision {prec} | AP = {ap:.3f}")            # 원본 셀 51

[no weight] recall 24/31 = 77.4% | precision 24/26 = 92.3% | AP = 0.869
[balanced ] recall 30/31 = 96.8% | precision 30/65 = 46.2% | AP = 0.871


**결과** · 예측 정답은 아님 — AP 0.869 vs 0.871

- `[no weight] recall 24/31 = 77.4% | precision 24/26 = 92.3% | AP = 0.869` — 놓침이 많음
- `[balanced ] recall 30/31 = 96.8% | precision 30/65 = 46.2% | AP = 0.871` — 놓침은 줄고 헛발질이 늘어남 · AP는 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>가중치 없음</td><td>다수 클래스 쪽으로 기움</td><td>오탐이 비쌀 때</td></tr>
<tr><td><code>class_weight="balanced"</code></td><td>소수 오분류 벌점 ↑ → 기준선 이동</td><td>놓침이 비쌀 때</td></tr>
<tr><td>AP</td><td>순위 능력 · 기준과 무관</td><td>가중치로 거의 안 변함</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 무엇을 놓치면 더 아픈지 먼저 정해야 이 선택을 평가할 수 있음 (파일별 4)

> ✅ **한 줄 정리** · class_weight='balanced'는 분류력(AP)을 키우기보다 판단 기준을 소수 클래스 쪽으로 옮겨 재현율과 정밀도를 맞바꾼다.

**딥러닝 코드에서는**

- `CrossEntropyLoss(weight=)` · `pos_weight` — 같은 효과

---
## 7. 종합 실습 · 여러 그루 다수결과 세 모델 회고

> **이 장의 흐름**
>
> - 개발 120개를 복원추출해 트리 6그루 다수결 × 20회 (Support F · 봉투는 다시 안 엶)
> - 그다음 로지스틱 · KNN · 트리를 같은 CV 계약으로 회고 비교 (Closing Support · 선택은 안 바꿈)

<img src="images/pfm6_practice.png" alt="단독 트리 120그루 vs 6그루 다수결 20회 정답 수 분포" width="599">

**파이프라인 위치** · ⑦ 모델 비교 → 앙상블(파일별 7)로

> 🎯 **6그루 다수결은 평균 점수를 올리는가, 흔들림을 줄이는가? 세 모델은 Iris에서 구분되는가?**

💡 **쉽게 말하면**

- 다수결은 **운 나쁜 트리를 뽑을 위험**을 줄임.
- 점수가 같으면 남는 차이는 **판단 규칙을 읽을 수 있는 형태**.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>seed 20번 × 복원추출 6그루</td><td>3-1 · 6-1</td></tr>
<tr><td>2</td><td>그루마다 · 다수결마다 정답 수</td><td>1-2</td></tr>
<tr><td>3</td><td>평균 · 표준편차 · 최저 · 최고</td><td>pandas</td></tr>
<tr><td>4</td><td>세 모델 같은 CV 계약</td><td>4-2 · 파일별 3</td></tr>
</tbody></table>

**먼저 예측** · 6그루 다수결의 표준편차는 단독 트리보다 클까 작을까? — 답은 코드 실행 결과 아래에

In [18]:
# 1 · 2단계: 복원추출 6그루 다수결을 20번 (개발 120개만 · 봉투는 다시 안 엶)
# 왜 · 트리 한 그루의 흔들림을 여러 그루의 다수결로 줄일 수 있는지 확인하기 위해
single_scores, ensemble_scores = [], []                   # 단독 · 다수결 정답 수
for seed in range(20):                                    # 20회
    rng = np.random.RandomState(seed)                     # 회차마다 다른 추출
    preds = []                                            # 6그루의 예측
    for _ in range(6):                                    # 6그루
        idx = rng.choice(len(Xd), len(Xd), replace=True)  # 복원추출
        m = DT(max_depth=3, random_state=RS)              # 같은 설정
        pred = m.fit(Xd[idx], yd[idx]).predict(Xd)        # 학습 · 예측
        preds.append(pred)                                # 모음
        single_scores.append(int((pred == yd).sum()))     # 단독 정답 수
    votes = np.vstack(preds)                              # 6 × 120
    major = np.array([np.bincount(col, minlength=3).argmax() for col in votes.T])  # 꽃마다 다수결
    ensemble_scores.append(int((major == yd).sum()))      # 다수결 정답 수


# 3단계: 요약
# 왜 · 단일 트리와 다수결의 평균 · 흔들림을 숫자로 비교하기 위해
stats = ["mean", "std", "min", "max"]                         # 요약 4개 (std는 표본 표준편차)
summ = pd.DataFrame({"단독 트리 120그루": pd.Series(single_scores).agg(stats),  # 단독
                     "6그루 다수결 20회": pd.Series(ensemble_scores).agg(stats)}).T  # 다수결
print(summ.round(2).to_string())                              # 원본 셀 61의 표
print("FINAL_HOLDOUT open count:", FINAL_HOLDOUT.open_count)  # 여전히 1

               mean   std    min    max
단독 트리 120그루  116.27  1.52  111.0  119.0
6그루 다수결 20회  117.35  0.81  116.0  119.0


FINAL_HOLDOUT open count: 1


**실행 흐름**

1. 로지스틱(+스케일러) · KNN(+스케일러) · 트리 세 모델 준비
2. 같은 개발 120개 · 같은 CV 계약으로 점수 출력
3. 고른 모델이 그대로인지 · 봉투를 연 횟수 확인

In [19]:
# 4단계: 세 모델을 같은 CV 계약으로 회고 비교 (선택은 바꾸지 않음)
# 왜 · 이미 고른 모델을 다른 모델과 같은 조건에서 돌아보며 비교하기 위해
from sklearn.linear_model import LogisticRegression  # 확률
from sklearn.neighbors import KNeighborsClassifier   # 거리
from sklearn.pipeline import make_pipeline           # 스케일러 + 모델
from sklearn.preprocessing import StandardScaler     # 조건 맞추기

lr3000 = LogisticRegression(max_iter=3000)                           # 원본 설정
trio = {"LogisticRegression(+Scaler)": make_pipeline(StandardScaler(), lr3000),  # 조건 맞추기
        "KNN k=5 (+Scaler)": make_pipeline(StandardScaler(), KNeighborsClassifier()),
        "DecisionTree(max_depth=3)": DT(max_depth=3, random_state=RS)}
for name, m in trio.items():                                         # 같은 개발 120 · 같은 CV
    c, mean, std = cv_report(m, Xd, yd)                              # 원본 셀 64
    print(f"{name:<28s} {c}/120 · mean {mean:.4f} · std {std:.4f}")  # 세 줄
# 규율 확인
print("선택 변경 없음:", best_dt is gs.best_estimator_, "· open count:", FINAL_HOLDOUT.open_count)

LogisticRegression(+Scaler)  115/120 · mean 0.9583 · std 0.0264


KNN k=5 (+Scaler)            115/120 · mean 0.9583 · std 0.0264
DecisionTree(max_depth=3)    115/120 · mean 0.9583 · std 0.0264
선택 변경 없음: True · open count: 1


**결과** · 예측 정답은 작음 — 1.52 → 0.81 (원본은 모표준편차로 1.52 · 0.79)

- `단독 트리 120그루  116.27  1.52  111.0  119.0` — 운 나쁘면 111
- `6그루 다수결 20회  117.35  0.81  116.0  119.0` — 최고는 같고 바닥이 올라옴 · 흔들림 절반
- `FINAL_HOLDOUT open count: 1` — 봉투는 다시 안 열림
- `LogisticRegression(+Scaler)  115/120 · mean 0.9583 · std 0.0264` — 세 모델이 표준편차까지 똑같음
- `DecisionTree(max_depth=3)    115/120 · mean 0.9583 · std 0.0264` — 이 CV로는 성적을 구분 못 함
- `선택 변경 없음: True · open count: 1` — 회고 비교일 뿐 · 최종 점수를 새로 만들지 않음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>로지스틱 회귀</td><td>계수 12개(3클래스 × 4) + 절편 3개</td><td>경계: 직선 · 비교적 안정</td></tr>
<tr><td>KNN</td><td>특정 예측의 이웃은 확인 · 전역 규칙 요약은 어려움</td><td>경계: 곡선 · k에 따라 변동</td></tr>
<tr><td>결정 트리</td><td>질문 경로 · 그림 한 장</td><td>경계: 축 평행 계단 · seed에 따라 규칙이 흔들림</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 어느 그루가 119짜리인지는 정답을 봐야 알 수 있음 → 정답을 보고 고르면 시험지를 보고 공부 범위를 정한 것
> - 다수결로 잃는 것 · 1장에서 손으로 읽던 규칙을 더 이상 그림 한 장으로 읽을 수 없음
> - 스케일링 · 고차원 · 학습 속도는 이 노트북에서 확인하지 않음 → Iris는 전부 cm라 스케일 차이가 작음

> ✅ **한 줄 정리** · 여러 그루의 다수결은 평균보다 흔들림(바닥)을 줄이고, 점수가 같을 때 남는 차이는 판단 규칙을 읽을 수 있는 형태다.

**딥러닝 코드에서는**

- 앙상블 · 체크포인트 평균 → 흔들림을 줄이는 대신 해석 가능성을 잃음 (파일별 7)

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 상자 · 경로 읽기</td><td>봉투 30개 봉인 · 루트 petal length ≤ 2.45 · 꽃 index 2 → 잎 [0, 39, 3] · 0.9286</td><td>1-1 ~ 1-3</td></tr>
<tr><td>2. 좋은 질문</td><td>부모 0.667 → A 0.333 vs B 0.531</td><td>2-1 · 2-2</td></tr>
<tr><td>3. 깊이 · 과적합</td><td>CV 80 → 113 → 115 → … 114 · sepal 경계 82/84 → 26/36 vs 69/84 → 31/36</td><td>3-1 · 3-2</td></tr>
<tr><td>4. 계약 · 봉투</td><td>공동 1위 16개 · cv만 바꾸면 4 → 3 · 봉투 29/30 (1건 = 3.3%p)</td><td>4-2 · 4-3 · 5-1</td></tr>
<tr><td>5. 중요도 · 흔들림</td><td>sepal 0.000이어도 31/36 · seed 30개 → 루트 2가지(22 · 8)</td><td>5-2 · 6-1</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>트리 상자</td><td>질문 · gini · samples · value · class</td><td>규칙 설명</td></tr>
<tr><td><code>A</code></td><td>Gini</td><td>1 − Σp² · 크기 감안 가중 평균</td><td>엔트로피 계열</td></tr>
<tr><td><code>A</code></td><td>max_depth</td><td>질문 횟수 · train과 CV가 따로 놂</td><td>층 · epoch</td></tr>
<tr><td><code>A</code></td><td>계약</td><td>후보 · 점수 · 분할 · 바뀌면 선택도 바뀜</td><td>seed · val 분할 명시</td></tr>
<tr><td><code>A</code></td><td>봉투 1회</td><td>선택 종료 뒤 · 1건 = 3.3%p</td><td>test loader 1회</td></tr>
<tr><td><code>A</code></td><td>중요도</td><td>이 트리의 비중 · 0 ≠ 쓸모없음 · 인과 아님</td><td>Grad-CAM 해석 주의</td></tr>
<tr><td><code>A</code></td><td>흔들림</td><td>seed만 바꿔도 루트가 바뀜</td><td>seed 명시</td></tr>
<tr><td><code>B</code></td><td>가지치기</td><td>복잡도 제한 · 성능 보장 아님</td><td>dropout</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>B</code></td><td>NaN · class_weight</td><td>1.3+ 직접 처리 · 기준선 이동</td><td>결측 마스크 · <code>pos_weight</code></td></tr>
<tr><td><code>B</code></td><td>다수결</td><td>흔들림(바닥)을 줄임 · 해석은 잃음</td><td>앙상블</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 봉투를 선택 종료 전에 열면?
- **1-2** · 상자의 value는 무엇인가?
- **1-3** · 꽃 index 2의 predict_proba는?
- **2-1** · A와 B 중 좋은 질문은?
- **2-2** · Iris 루트 Gini는?
- **3-1** · 깊이를 늘리면?
- **3-2** · sepal 경계에서 깊이 제한 없음의 결과는?
- **4-1** · 가지치기를 더하면?
- **4-2** · best_params_는 유일한 정답인가?
- **4-3** · cv만 바꾸면?
- **5-1** · 봉투 29/30이 CV 95.8%보다 높은 의미는?
- **5-2** · 중요도 0의 뜻은?
- **6-1** · seed만 바꾼 30그루의 루트 질문은?
- **6-2** · Gini vs Entropy?
- **6-3** · sklearn 트리는 NaN을 직접 처리하나?
- **6-4** · class_weight='balanced'의 효과는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `DecisionTreeClassifier(max_depth=2, criterion="entropy", random_state=RANDOM_STATE)`를 개발 120개에 fit해 루트 질문 출력
2. 그 트리의 잎 개수 출력

In [20]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2에서 후보 C `petal width <= 1.75`의 크기 감안 Gini는? A(0.333)보다 좋을까?
2. 3-2에서 깊이를 2로 바꾸면 unseen 정답 수는?
3. 7장에서 6그루 대신 25그루로 다수결하면 표준편차는?

In [21]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · RuntimeError · 선택을 먼저 종료해야 함
- **1-2** · 클래스별 개수 (알파벳순) · 예 [0, 39, 3]
- **1-3** · [0, 0.9286, 0.0714] = 잎 비율 39 ÷ 42
- **2-1** · A · 왼쪽 40/40 · B는 오른쪽 95개가 크게 섞임
- **2-2** · 0.667 (3클래스 최대 1 − 1/3)
- **3-1** · train은 1.0까지 · CV는 115에서 멈춤 (과적합)
- **3-2** · train 82/84 · unseen 26/36 (깊이 3은 31/36)
- **4-1** · 잎 7 → 5 · CV 113 그대로 (복잡도 제한 도구)
- **4-2** · 아님 · 공동 1위 16개 중 순서상 먼저
- **4-3** · max_depth 4 → 3 · 결과는 계약과 함께 공유
- **5-1** · 30개에서 1건 = 3.3%p 흔들림 · 다시 고르지 않음
- **5-2** · 이 트리의 분기에 안 쓰였다 · 쓸모없다가 아님
- **6-1** · 2가지 · petal width 22번 · petal length 8번
- **6-2** · 115 vs 114 · 1건 차이 · 기본값 Gini
- **6-3** · 1.3부터 기본 설정에서 처리 · 우열은 데이터마다
- **6-4** · 재현율 ↑ · 정밀도 ↓ · AP는 거의 같음 (기준선 이동)

#### 코드 문제 답

```python
m_e = DT(max_depth=2, criterion="entropy", random_state=RS)
m_e.fit(Xd, yd)
print(describe_root(m_e))  # petal length <= 2.45
print(m_e.get_n_leaves())  # 3
```

#### 변형 문제 답

*문제 → 답*

- **1** · 0.393 — A(0.333)보다 나쁨 · setosa를 한쪽에 완전히 모으지 못함
- **2** · unseen 31/36 — 깊이 3과 같음 · 이 분할에서는 깊이 2도 충분
- **3** · 평균 118.05 · 표준편차 0.39 — 그루가 많을수록 흔들림이 더 줄어듦 · 대신 학습 · 예측 비용 증가

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 중요도 0인 sepal을 지워도 된다고 답함</td><td>중요도를 '정보량'으로 착각</td><td>중요도는 이 트리가 쓴 비중</td><td>그 피처만으로 학습하면 몇 점인가?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 결정 트리는 어떻게 분할 기준을 고르나요?**
> - 가능한 모든 피처와 기준값으로 데이터를 둘로 나눠 보고, 양쪽의 불순도를 개수 비율로 가중 평균했을 때 부모보다 가장 많이 낮추는 질문을 고릅니다.
> - 기본 불순도는 Gini로 1에서 클래스 비율 제곱합을 뺀 값입니다.
> - Iris에서는 부모 0.667을 0.333으로 낮춘 petal length 2.45가 뽑혔습니다.

**Q2. 결정 트리의 과적합은 어떻게 막나요?**
> - max_depth로 질문 횟수를 제한하고 min_samples_split이나 min_samples_leaf로 너무 작은 노드를 막습니다.
> - 다만 가지치기는 성능 보장이 아니라 복잡도 제한이라 효과는 같은 교차검증 계약으로 확인합니다.
> - Iris에서는 깊이를 늘려도 train만 1.0까지 오르고 CV는 115에서 멈췄습니다.

**Q3. feature_importances_가 0이면 그 피처를 지워도 되나요?**
> - 아닙니다.
> - 중요도는 그 특정 트리가 분기에서 줄인 불순도의 비중일 뿐이라 0은 이번 트리에서 안 쓰였다는 뜻입니다.
> - Iris에서 중요도 0인 sepal 두 개만으로도 처음 보는 꽃의 86%를 맞혔습니다.
> - 또 seed만 바꿔도 트리 규칙과 중요도가 달라지므로 random_state와 함께 보고합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>DecisionTreeClassifier</code></td><td>sklearn.tree</td><td>결정 트리</td><td>1-2 <code>DecisionTreeClassifier(max_depth=2, …)</code></td></tr>
<tr><td><code>export_text</code></td><td>sklearn.tree</td><td>트리를 글자로</td><td>1-2 <code>export_text(shallow_tree, …)</code></td></tr>
<tr><td><code>tree_.value</code> · <code>impurity</code> · <code>n_node_samples</code></td><td>트리 내부 구조</td><td>상자 정보</td><td>1-2 <code>t.value[i][0] * t.n_node_samples[i]</code></td></tr>
<tr><td><code>apply</code> · <code>decision_path</code></td><td>결정 트리 메서드</td><td>도착 잎 · 경로</td><td>1-3 <code>shallow_tree.decision_path(sample)</code></td></tr>
<tr><td><code>np.bincount</code></td><td>numpy</td><td>클래스별 개수</td><td>2-1 <code>np.bincount(y_development[mask], minlength=3)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>cross_val_predict</code> · <code>cross_val_score</code></td><td>sklearn.model_selection</td><td>CV 정답 수 · 점수</td><td>3-1 <code>cross_val_predict(model, Xv, yv, cv=CV)</code></td></tr>
<tr><td><code>min_samples_split</code> · <code>min_samples_leaf</code></td><td>결정 트리 파라미터</td><td>가지치기</td><td>4-1 <code>min_samples_split=8, min_samples_leaf=4</code></td></tr>
<tr><td><code>GridSearchCV</code> · <code>rank_test_score</code></td><td>sklearn.model_selection</td><td>후보 비교 · 공동 1위</td><td>4-2 <code>GridSearchCV(…, cv=CV)</code></td></tr>
<tr><td><code>feature_importances_</code></td><td>결정 트리 속성</td><td>이 트리의 불순도 감소 비중</td><td>5-2 <code>best_dt.feature_importances_</code></td></tr>
<tr><td><code>criterion="entropy"</code></td><td>결정 트리 파라미터</td><td>분할 기준</td><td>6-2 <code>criterion=crit</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>class_weight="balanced"</code></td><td>결정 트리 파라미터</td><td>소수 클래스 가중</td><td>6-4 <code>**kw</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>상자의 value를 개수로 읽기</td><td>잎 비율을 100% 확신으로 읽기</td></tr>
<tr><td>양쪽을 크기까지 감안해 비교</td><td>작은 그룹의 깨끗함에 속기</td></tr>
<tr><td>결과는 계약과 함께</td><td>best_params_를 유일한 정답으로</td></tr>
<tr><td>봉투는 선택 종료 뒤 1회</td><td>봉투 점수를 보고 다시 고르기</td></tr>
<tr><td>중요도는 이 트리 · 이 seed 기준</td><td>중요도 0인 피처 지우기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>Gini vs Entropy</td><td>1 − Σp² vs −Σp log p</td><td>기본은 Gini</td></tr>
<tr><td>split vs leaf</td><td>나누기 전 멈춤 vs 나눠 본 뒤 다른 분할</td><td>leaf가 더 부드러운 제한</td></tr>
<tr><td>train vs CV 정답률</td><td>외운 문제 vs 처음 보는 문제</td><td>선택은 CV</td></tr>
<tr><td>중요도 0 vs 쓸모없음</td><td>이 트리에서 안 씀 vs 정보 없음</td><td>같지 않음</td></tr>
<tr><td>단독 트리 vs 다수결</td><td>읽힘 · 흔들림 vs 안정 · 안 읽힘</td><td>목적에 따라</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 2-2 Gini를 손으로 다시 계산</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>1-2 경로 따라가기를 다른 꽃으로 · 7장 다수결을 그루 수만 바꿔 다시</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 7(앙상블) 시작 전 ✅ 줄만 읽기</td><td>15분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 Act 1 ~ 7을 1 ~ 6장 · 16개 섹션으로 · Support A ~ E는 쓰는 장 안에, F와 Closing은 7장 종합 실습으로
- 원본의 긴 도우미 함수 6개는 쓰는 섹션 안에 짧게 풂
- (show_tree · leaf_table · cv_report · split_count_rate · trace_one_flower · describe_root)
- 원본 그림 6개(트리 · 깊이 곡선 · 혼동행렬 · 트리 · 중요도 · 경계 · PR · 분포)는 같은 계산으로 장 그림 · 결과 줄로
- 원본 Prediction 정답의 접힌 블록(details 태그)은 형식 규칙상 각 섹션의 '결과' 줄로 옮김

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 0 · 1</td><td>오늘의 질문 · 할 수 있는 것 · 흐름 · 복습</td><td>0. 원본 목표</td></tr>
<tr><td>셀 2 ~ 5</td><td>준비 · 봉투 · 도우미 함수</td><td>0. 준비 · 1-1 · 각 섹션</td></tr>
<tr><td>셀 6 ~ 8</td><td>Act 1 · 얕은 트리 · 상자 읽기</td><td>1-2 · 1장 그림</td></tr>
<tr><td>셀 8 ~ 11</td><td>Prediction 1 · 꽃 한 송이</td><td>1-3</td></tr>
<tr><td>셀 12 ~ 14</td><td>Act 2 · 질문 A vs B</td><td>2-1 · 2장 그림</td></tr>
<tr><td>셀 15 ~ 17</td><td>불순도 · Gini 공식 · 크기 감안</td><td>2-2</td></tr>
<tr><td>셀 18 ~ 22</td><td>Act 3 · 깊이별 train vs CV</td><td>3-1 · 3장 그림</td></tr>
<tr><td>셀 55 ~ 59</td><td>Support E · 경계 모양</td><td>3-2</td></tr>
<tr><td>셀 23 ~ 26</td><td>Act 4 · 가지치기</td><td>4-1</td></tr>
<tr><td>셀 26 ~ 29</td><td>GridSearchCV · 공동 1위 · 선택 종료</td><td>4-2</td></tr>
<tr><td>셀 52 ~ 54</td><td>Support D · 계약</td><td>4-3 · 4장 그림</td></tr>
<tr><td>셀 30 ~ 34</td><td>Act 5 · 봉투 열기</td><td>5-1</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 35 ~ 38</td><td>Act 6 · 중요도</td><td>5-2 · 5장 그림</td></tr>
<tr><td>셀 39 ~ 42</td><td>Act 7 · 흔들림</td><td>6-1 · 6장 그림</td></tr>
<tr><td>셀 46 ~ 48</td><td>Support A · Gini vs Entropy</td><td>6-2</td></tr>
<tr><td>셀 48 ~ 50</td><td>Support B · 결측치</td><td>6-3</td></tr>
<tr><td>셀 50 ~ 52</td><td>Support C · 불균형</td><td>6-4</td></tr>
<tr><td>셀 60 ~ 62</td><td>Support F · 다수결</td><td>7. 종합 실습 1 ~ 3단계</td></tr>
<tr><td>셀 63 ~ 66</td><td>Closing Support · 세 모델 · 비교표</td><td>7. 종합 실습 4단계 · 비교 표</td></tr>
<tr><td>셀 43 ~ 45</td><td>오늘의 정리 · 회귀 트리 · 기억할 흐름</td><td>그래서 오늘 · 최종 요약표</td></tr>
</tbody></table>

### 전체 코드

1 ~ 7장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 최종 데이터 봉인
2. 1-2 얕은 트리 학습
3. 1-3 한 송이의 경로 따라가기

In [22]:
# ---------- 1-1 ----------
# 왜 · 개발용과 최종 평가용을 처음에 한 번만 나누기 위해
X_development, _Xh, y_development, _yh = train_test_split(  # 개발 120 · 봉투 30
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)


# 왜 · 최종 평가 데이터를 실수로 두 번 쓰는 일을 코드로 막기 위해
class FinalHoldoutVault:                          # 선택이 끝난 뒤 단 한 번만 열 수 있는 평가 봉투
    def __init__(self, X_value, y_value):         # 봉투에 넣기
        self.__X, self.__y = X_value.copy(), y_value.copy()  # 밖에서 못 보게 숨김
        self._frozen, self.open_count = False, 0  # 선택 종료 여부 · 연 횟수

    def freeze_selection(self):  # 선택 종료 선언
        self._frozen = True      # 이제 열 수 있음

    # 왜 · 선택을 끝내기 전에는 열 수 없고, 연 뒤에는 다시 열 수 없게 하기 위해
    def open_once(self):                                         # 한 번만 열기
        if not self._frozen:                                     # 선택이 안 끝났으면
            raise RuntimeError("모델 선택을 먼저 종료해야 함.")  # 막음
        if self.open_count:                                      # 이미 열었으면
            raise RuntimeError("FINAL_HOLDOUT은 이미 열었음.")   # 막음
        self.open_count += 1                                     # 연 횟수 기록
        return self.__X.copy(), self.__y.copy()                  # 꺼내 줌


# 왜 · 원래 변수를 지워 봉투 밖에서 최종 데이터에 손대지 못하게 하기 위해
FINAL_HOLDOUT = FinalHoldoutVault(_Xh, _yh)        # 봉인
del _Xh, _yh                                       # 원본 변수도 지움 (실수로 못 쓰게)
print(f"DEVELOPMENT_POOL {len(X_development)}개 / FINAL_HOLDOUT 30개 봉인")  # 120 / 30
print(f"open count = {FINAL_HOLDOUT.open_count}")  # 0 확인
Xd, yd = X_development, y_development              # 개발 120개의 짧은 이름
try:                                               # 선택 종료 전에 열어 보기
    FINAL_HOLDOUT.open_once()                      # 아직 freeze 안 함
except RuntimeError as e:                          # 막힘
    print("열기 시도:", e)                         # 안내 문구


# ---------- 1-2 ----------
# 왜 · 깊이 2짜리 트리로 질문 구조를 눈으로 읽을 수 있게 하기 위해
shallow_tree = DT(max_depth=2, random_state=RS)                                  # 질문 최대 2번
shallow_tree.fit(Xd, yd)                                                         # 개발 120개로만
names_short = [n.replace(" (cm)", "") for n in feature_names]                    # 짧은 이름
print(export_text(shallow_tree, feature_names=names_short, show_weights=True))   # 트리를 글자로
t = shallow_tree.tree_                                                           # 내부 구조
leaves = [i for i in range(t.node_count) if t.children_left[i] == -1]            # 잎 번호
for i in leaves:                                                                 # 잎마다
    counts = (t.value[i][0] * t.n_node_samples[i]).round().astype(int).tolist()  # 비율 × 개수
    # 상자 정보
    print(f"잎 {i}: gini {t.impurity[i]:.4f} · samples {t.n_node_samples[i]} · value {counts}")


# ---------- 1-3 ----------
# 왜 · 꽃 한 송이가 어떤 질문을 거쳐 잎에 도착하는지 직접 따라가 보기 위해
sample = Xd[[2]]                                                        # 세 번째 꽃 (2차원 유지)
print("꽃 치수:", dict(zip(names_short, sample[0].round(2).tolist())))  # 측정값 4개
leaf = shallow_tree.apply(sample)[0]                                    # 도착 잎 번호
for node in shallow_tree.decision_path(sample).indices:                 # 지나간 상자들
    if node == leaf:                                                    # 잎은 질문이 없음
        continue                                                        # 건너뜀
    fi, th = t.feature[node], t.threshold[node]                         # 질문 피처 · 기준값
    go = "예, 왼쪽" if sample[0, fi] <= th else "아니오, 오른쪽"        # 답
    print(f"  {names_short[fi]} <= {th:.2f}?  관측 {sample[0, fi]:.2f}  →  {go}")  # 경로 한 줄
print("  예측:", target_names[shallow_tree.predict(sample)[0]],         # 다수 클래스
      "| predict_proba:", shallow_tree.predict_proba(sample)[0].round(4).tolist())  # 잎 비율

DEVELOPMENT_POOL 120개 / FINAL_HOLDOUT 30개 봉인
open count = 0
열기 시도: 모델 선택을 먼저 종료해야 함.
|--- petal length <= 2.45
|   |--- weights: [40.00, 0.00, 0.00] class: 0
|--- petal length >  2.45
|   |--- petal width <= 1.65
|   |   |--- weights: [0.00, 39.00, 3.00] class: 1
|   |--- petal width >  1.65
|   |   |--- weights: [0.00, 1.00, 37.00] class: 2

잎 1: gini 0.0000 · samples 40 · value [40, 0, 0]
잎 3: gini 0.1327 · samples 42 · value [0, 39, 3]
잎 4: gini 0.0512 · samples 38 · value [0, 1, 37]
꽃 치수: {'sepal length': 6.8, 'sepal width': 2.8, 'petal length': 4.8, 'petal width': 1.4}
  petal length <= 2.45?  관측 4.80  →  아니오, 오른쪽
  petal width <= 1.65?  관측 1.40  →  예, 왼쪽
  예측: versicolor | predict_proba: [0.0, 0.9286, 0.0714]


**실행 흐름**

1. 2-1 질문별 나눔 결과
2. 2-2 Gini로 질문 비교

In [23]:
# ---------- 2-1 ----------
# 왜 · 질문 하나가 데이터를 얼마나 깨끗하게 나누는지 개수 · 비율로 보기 위해
def split_count_rate(fi, th):                                # 질문 하나 → 양쪽 품종 개수
    rows = []                                                # 예 · 아니오 두 줄
    for side, mask in [("예(왼쪽)", Xd[:, fi] <= th),        # 기준 이하
                       ("아니오(오른쪽)", Xd[:, fi] > th)]:  # 기준 초과
        c = np.bincount(yd[mask], minlength=3)               # 품종별 개수
        rows.append([side, int(mask.sum()), *c.tolist(), round(c.max() / mask.sum(), 4)])  # 한 줄
    return pd.DataFrame(rows, columns=["답", "전체", *target_names, "가장 큰 Rate"])  # 표


# 왜 · 좋은 질문과 나쁜 질문의 나눔 결과를 나란히 비교하기 위해
print("A: petal length <= 2.45")                         # 후보 A
print(split_count_rate(2, 2.45).to_string(index=False))  # 원본 셀 13
print("B: sepal length <= 5.00")                         # 후보 B
print(split_count_rate(0, 5.00).to_string(index=False))  # 원본 셀 13


# ---------- 2-2 ----------
# 왜 · 트리가 질문을 고르는 기준인 Gini 불순도를 직접 계산해 보기 위해
def gini(labels):                                       # 1 - Σ p²
    p = np.bincount(labels, minlength=3) / len(labels)  # 품종 비율
    return 1 - (p ** 2).sum()                           # 섞일수록 큼


# 왜 · 자식 그룹 크기를 반영한 가중 Gini로 질문의 좋고 나쁨을 숫자로 비교하기 위해
def weighted_child_gini(fi, th):                    # 양쪽을 개수 비율로
    left = Xd[:, fi] <= th                          # 예 그룹
    n, nl = len(yd), left.sum()                     # 전체 · 왼쪽 개수
    gl, gr = gini(yd[left]), gini(yd[~left])        # 양쪽 Gini
    return gl, gr, nl / n * gl + (n - nl) / n * gr  # 왼 · 오 · 가중 평균


# 왜 · 질문 전후로 불순도가 얼마나 줄었는지 확인하기 위해
print(f"부모 Gini: {gini(yd):.3f}")                                        # 3클래스 최대
for tag, fi, th in [("A", 2, 2.45), ("B", 0, 5.00)]:                       # 두 후보
    gl, gr, gw = weighted_child_gini(fi, th)                               # 계산
    print(f"{tag}: 왼쪽 {gl:.3f} · 오른쪽 {gr:.3f} · 크기 감안 {gw:.3f}")  # 원본 셀 17 표

A: petal length <= 2.45
       답  전체  setosa  versicolor  virginica  가장 큰 Rate
   예(왼쪽)  40      40           0          0        1.0
아니오(오른쪽)  80       0          40         40        0.5
B: sepal length <= 5.00
       답  전체  setosa  versicolor  virginica  가장 큰 Rate
   예(왼쪽)  25      23           1          1     0.9200
아니오(오른쪽)  95      17          39         39     0.4105
부모 Gini: 0.667
A: 왼쪽 0.000 · 오른쪽 0.500 · 크기 감안 0.333
B: 왼쪽 0.150 · 오른쪽 0.631 · 크기 감안 0.531


**실행 흐름**

1. 3-1 깊이별 CV
2. 3-2 과적합 눈으로 보기

In [24]:
# ---------- 3-1 ----------
# 왜 · 모든 깊이를 같은 CV 계약으로 비교해 공정하게 고르기 위해
def cv_report(model, Xv, yv):                         # 원본 도우미: 맞힌 개수 · 평균 · 표준편차
    oof = cross_val_predict(model, Xv, yv, cv=CV)     # fold마다 처음 보는 예측
    s = cross_val_score(model, Xv, yv, cv=CV)         # fold별 정답률
    return int((oof == yv).sum()), s.mean(), s.std()  # 세 값


rows = []                                                                   # 깊이마다
for depth in [1, 2, 3, 4, 5, 8, None]:                                      # 원본 셀 19의 후보
    m = DT(max_depth=depth, random_state=RS)                                # 질문 횟수 제한
    c, mean, std = cv_report(m, Xd, yd)                                     # 처음 보는 문제
    m.fit(Xd, yd)                                                           # 외운 문제용 학습
    tr = accuracy_score(yd, m.predict(Xd))                                  # train 정답률
    rows.append([str(depth), tr, c, mean, std, m.get_n_leaves()])           # 한 줄
cols = ["max_depth", "train", "cv_correct", "cv_mean", "cv_std", "leaves"]  # 원본 열 이름
depth_df = pd.DataFrame(rows, columns=cols)                                 # 깊이별 표
print(depth_df.round(4).to_string(index=False))                             # 원본 표


# ---------- 3-2 ----------
# 왜 · 특성 두 개만 써서 깊은 트리가 학습 데이터를 외우는 과적합을 눈으로 보기 위해
X2 = Xd[:, [0, 1]]                                                     # sepal length · width만
X2_tr, X2_un, y2_tr, y2_un = train_test_split(                         # 84 / 36 (봉투는 그대로)
    X2, yd, test_size=0.30, stratify=yd, random_state=RS)
dt_free = DT(random_state=RS).fit(X2_tr, y2_tr)                        # 제한 없음
dt_3 = DT(max_depth=3, random_state=RS).fit(X2_tr, y2_tr)              # 깊이 3
for label, m in [("max_depth=None", dt_free), ("max_depth=3", dt_3)]:  # 두 트리
    tr_ok, un_ok = (m.predict(X2_tr) == y2_tr).sum(), (m.predict(X2_un) == y2_un).sum()  # 정답 수
    # 원본 셀 56
    print(f"[{label}] 잎 {m.get_n_leaves():2d} | train {tr_ok}/84 | unseen {un_ok}/36")

max_depth  train  cv_correct  cv_mean  cv_std  leaves
        1 0.6667          80   0.6667  0.0000       2
        2 0.9667         113   0.9417  0.0204       3
        3 0.9833         115   0.9583  0.0264       5
        4 0.9917         113   0.9417  0.0204       7
        5 1.0000         114   0.9500  0.0167       8
        8 1.0000         114   0.9500  0.0167       8
     None 1.0000         114   0.9500  0.0167       8
[max_depth=None] 잎 31 | train 82/84 | unseen 26/36
[max_depth=3] 잎  7 | train 69/84 | unseen 31/36


**실행 흐름**

1. 4-1 가지치기
2. 4-2 GridSearchCV
3. 4-3 CV 계약 비교

In [25]:
# ---------- 4-1 ----------
# 왜 · 가지치기 조건이 복잡도를 줄이는지와 점수가 어떻게 되는지 함께 확인하기 위해
pruned = DT(max_depth=4, min_samples_split=8, min_samples_leaf=4, random_state=RS)  # 가지치기 추가
# 두 후보
cands = {"depth=4 only": DT(max_depth=4, random_state=RS), "depth=4 + split/leaf": pruned}
for label, m in cands.items():                                                      # 두 후보
    c, mean, std = cv_report(m, Xd, yd)                                             # 같은 CV 계약
    leaves_n = m.fit(Xd, yd).get_n_leaves()                                         # 잎 수
    # 원본 셀 24
    print(f"{label:<22s} cv {c}/120 · mean {mean:.4f} · std {std:.4f} · 잎 {leaves_n}")


# ---------- 4-2 ----------
# 왜 · 여러 가지치기 조합을 같은 CV로 자동 탐색하기 위해
param_grid = {"max_depth": [2, 3, 4, None], "min_samples_split": [2, 4, 8],  # 원본 셀 27
              "min_samples_leaf": [1, 2, 4]}                                 # 4 × 3 × 3 = 36
gs = GridSearchCV(DT(random_state=RS), param_grid,                           # 같은 계약
                  cv=CV, scoring="accuracy").fit(Xd, yd)
n_tied = int((pd.DataFrame(gs.cv_results_)["rank_test_score"] == 1).sum())   # 공동 1위 수
print("선택된 후보:", gs.best_params_)                                       # 동점이면 순서상 먼저
print(f"CV mean accuracy: {gs.best_score_:.4f} · 36개 조합 중 공동 1위: {n_tied}개")  # 원본 셀 27
best_dt = gs.best_estimator_  # 개발 120개로 다시 학습된 트리
FINAL_HOLDOUT.freeze_selection()  # 이제 선택을 되돌리는 코드는 없음
print("SELECTION FROZEN · open count =", FINAL_HOLDOUT.open_count)           # 아직 0


# ---------- 4-3 ----------
# 왜 · CV 설정이 달라지면 점수도 달라진다는 것을 보여 같은 계약을 지켜야 하는 이유를 확인하기 위해
contracts = [("이전: cv=5 (shuffle 없음)", 5),                               # 섞지 않은 5등분
             ("이번: StratifiedKFold(5, shuffle=True, 42)", CV)]             # 오늘의 CV
for label, cv_obj in contracts:                                              # 분할 방식만 다름
    s = GridSearchCV(DT(random_state=RS), param_grid,                        # 후보 · 지표 같음
                     cv=cv_obj, scoring="accuracy").fit(Xd, yd)
    tied = int((pd.DataFrame(s.cv_results_)["rank_test_score"] == 1).sum())  # 공동 1위
    d_best = s.best_params_["max_depth"]                                     # 고른 깊이
    # 원본 셀 53
    print(f"{label} → max_depth = {d_best}, CV = {s.best_score_:.4f}, 공동 1위 = {tied}개")

depth=4 only           cv 113/120 · mean 0.9417 · std 0.0204 · 잎 7
depth=4 + split/leaf   cv 113/120 · mean 0.9417 · std 0.0204 · 잎 5


선택된 후보: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
CV mean accuracy: 0.9583 · 36개 조합 중 공동 1위: 16개
SELECTION FROZEN · open count = 0


이전: cv=5 (shuffle 없음) → max_depth = 4, CV = 0.9417, 공동 1위 = 2개


이번: StratifiedKFold(5, shuffle=True, 42) → max_depth = 3, CV = 0.9583, 공동 1위 = 16개


**실행 흐름**

1. 5-1 최종 데이터 1회 채점
2. 5-2 특성 중요도

In [26]:
# ---------- 5-1 ----------
# 왜 · 선택이 끝난 모델로 최종 데이터를 한 번만 채점하기 위해
X_final, y_final = FINAL_HOLDOUT.open_once()                            # 선택 종료 뒤 단 한 번
y_pred_final = best_dt.predict(X_final)                                 # 고른 트리 그대로
final_correct = int((y_pred_final == y_final).sum())                    # 맞힌 개수
print(f"FINAL_HOLDOUT: {final_correct}/30 = {final_correct / 30:.1%}")  # 원본 셀 31
print(f"참고: 30개 평가에서 1건 = {1 / 30:.1%}p · development CV: {gs.best_score_:.1%}")  # 비교
# 어디서 틀렸나
print("혼동행렬 (행 = 실제, 열 = 예측):", confusion_matrix(y_final, y_pred_final).tolist())
print(f"open count = {FINAL_HOLDOUT.open_count}")                       # 1
del X_final, y_final, y_pred_final                                      # 다시 못 쓰게 지움


# ---------- 5-2 ----------
# 왜 · 모델이 어떤 특성을 많이 썼는지 확인해 결과를 설명하기 위해
imp = pd.Series(best_dt.feature_importances_, index=names_short)  # 피처별 중요도
imp = imp.sort_values(ascending=False)                            # 큰 순서
print(imp.round(6).to_string())                                   # 원본 셀 36
print("합계:", round(imp.sum(), 6))                               # 정규화돼 1
# 분기에 쓴 것
print("이 트리가 쓴 피처:", [names_short[i] for i in sorted(set(best_dt.tree_.feature)) if i >= 0])

FINAL_HOLDOUT: 29/30 = 96.7%
참고: 30개 평가에서 1건 = 3.3%p · development CV: 95.8%
혼동행렬 (행 = 실제, 열 = 예측): [[10, 0, 0], [0, 9, 1], [0, 0, 10]]
open count = 1
petal length    0.579077
petal width     0.420923
sepal width     0.000000
sepal length    0.000000
합계: 1.0
이 트리가 쓴 피처: ['petal length', 'petal width']


**실행 흐름**

1. 6-1 seed별 첫 질문
2. 6-2 Gini vs Entropy
3. 6-3 · 6-4 결측 · 불균형

In [27]:
# 왜 · 결측이 있는 데이터에서 트리를 쓰는 방법을 시험할 가짜 데이터를 만들기 위해
from sklearn.datasets import make_classification     # 가짜 분류 데이터
from sklearn.impute import SimpleImputer             # 평균으로 채우기
# 왜 · 불균형 데이터에서는 정확도보다 AP가 더 정직하다는 것을 보이기 위해
from sklearn.metrics import average_precision_score  # PR 곡선 요약


# ---------- 6-1 ----------
# 왜 · seed가 바뀌면 첫 질문도 바뀔 수 있다는 트리의 불안정성을 보기 위해
def describe_root(m):                                                # 루트 질문을 글자로
    tt = m.tree_                                                     # 내부 구조
    return f"{names_short[tt.feature[0]]} <= {tt.threshold[0]:.2f}"  # 첫 질문


roots = [describe_root(DT(random_state=s).fit(Xd, yd))   # 같은 데이터
         for s in range(30)]                             # seed만 0 ~ 29
print(pd.Series(roots).value_counts().to_string())       # 원본 셀 40
print("데이터는 단 한 개도 바뀌지 않았음 (같은 120개)")  # 확인


# ---------- 6-2 ----------
# 왜 · 분할 기준을 바꿔도 결과가 크게 다르지 않은지 비교하기 위해
for name, crit in [("Gini", "gini"), ("Entropy", "entropy")]:  # 기준 두 개
    m = DT(max_depth=3, criterion=crit, random_state=RS)       # 깊이는 같음
    c, mean, _ = cv_report(m, Xd, yd)                          # 같은 CV
    print(f"{name:8s}: {c}/120 = {mean:.1%}")                  # 원본 셀 47


# ---------- 6-3 ----------

Xm, ym = make_classification(n_samples=300, n_features=4, n_informative=3,       # 원본 셀 49
                             n_redundant=0, random_state=RS)
Xm[np.random.RandomState(RANDOM_STATE).rand(*Xm.shape) < 0.10] = np.nan          # 10% 결측
Xtr, Xte, ytr, yte = train_test_split(                                           # 7 : 3
    Xm, ym, test_size=0.3, stratify=ym, random_state=RS)
native = DT(max_depth=3, random_state=RS).fit(Xtr, ytr)                          # NaN 그대로
imp = SimpleImputer(strategy="mean")                                             # train 평균으로
Xtr_f = imp.fit_transform(Xtr)  # train 평균으로 채운 학습 데이터
filled = DT(max_depth=3, random_state=RS).fit(Xtr_f, ytr)                        # 채운 데이터로
print(f"결측 {int(np.isnan(Xm).sum())}개 / 전체 {Xm.size}개")                    # 결측 수
print(f"NaN 그대로 학습     : {(native.predict(Xte) == yte).sum()}/{len(yte)}")  # 직접 처리
# 채움
print(f"SimpleImputer(mean) : {(filled.predict(imp.transform(Xte)) == yte).sum()}/{len(yte)}")


# ---------- 6-4 ----------

Xi, yi = make_classification(n_samples=1000, n_features=4,  # 정상 90 : 이상 10
                             n_informative=3, n_redundant=0, weights=[0.9, 0.1], random_state=RS)
Xtr2, Xte2, ytr2, yte2 = train_test_split(                                            # 7 : 3
    Xi, yi, test_size=0.3, stratify=yi, random_state=RS)
for label, kw in [("no weight", {}), ("balanced", {"class_weight": "balanced"})]:     # 두 설정
    m = DT(max_depth=3, random_state=RS, **kw).fit(Xtr2, ytr2)                        # 깊이 3
    pred, proba = m.predict(Xte2), m.predict_proba(Xte2)[:, 1]                        # 예측 · 확률
    # 칸 수
    tp, n_pos, n_pp = int(((pred == 1) & (yte2 == 1)).sum()), int(yte2.sum()), int(pred.sum())
    ap = average_precision_score(yte2, proba)                                         # 순위 요약
    rec, prec = f"{tp}/{n_pos} = {tp / n_pos:.1%}", f"{tp}/{n_pp} = {tp / n_pp:.1%}"  # 분수 · 비율
    print(f"[{label:9s}] recall {rec} | precision {prec} | AP = {ap:.3f}")            # 원본 셀 51

petal width <= 0.80     22
petal length <= 2.45     8
데이터는 단 한 개도 바뀌지 않았음 (같은 120개)
Gini    : 115/120 = 95.8%
Entropy : 114/120 = 95.0%
결측 128개 / 전체 1200개
NaN 그대로 학습     : 76/90
SimpleImputer(mean) : 76/90
[no weight] recall 24/31 = 77.4% | precision 24/26 = 92.3% | AP = 0.869
[balanced ] recall 30/31 = 96.8% | precision 30/65 = 46.2% | AP = 0.871


**실행 흐름**

1. 복원추출 다수결 20번
2. 평균 · 흔들림 요약
3. 세 모델 회고 비교

In [28]:
# ---------- 7. 종합 실습 ----------
from sklearn.linear_model import LogisticRegression  # 확률
from sklearn.neighbors import KNeighborsClassifier   # 거리
from sklearn.pipeline import make_pipeline           # 스케일러 + 모델
from sklearn.preprocessing import StandardScaler     # 조건 맞추기


# 1 · 2단계: 복원추출 6그루 다수결을 20번 (개발 120개만 · 봉투는 다시 안 엶)
# 왜 · 트리 한 그루의 흔들림을 여러 그루의 다수결로 줄일 수 있는지 확인하기 위해
single_scores, ensemble_scores = [], []                   # 단독 · 다수결 정답 수
for seed in range(20):                                    # 20회
    rng = np.random.RandomState(seed)                     # 회차마다 다른 추출
    preds = []                                            # 6그루의 예측
    for _ in range(6):                                    # 6그루
        idx = rng.choice(len(Xd), len(Xd), replace=True)  # 복원추출
        m = DT(max_depth=3, random_state=RS)              # 같은 설정
        pred = m.fit(Xd[idx], yd[idx]).predict(Xd)        # 학습 · 예측
        preds.append(pred)                                # 모음
        single_scores.append(int((pred == yd).sum()))     # 단독 정답 수
    votes = np.vstack(preds)                              # 6 × 120
    major = np.array([np.bincount(col, minlength=3).argmax() for col in votes.T])  # 꽃마다 다수결
    ensemble_scores.append(int((major == yd).sum()))      # 다수결 정답 수


# 3단계: 요약
# 왜 · 단일 트리와 다수결의 평균 · 흔들림을 숫자로 비교하기 위해
stats = ["mean", "std", "min", "max"]                         # 요약 4개 (std는 표본 표준편차)
summ = pd.DataFrame({"단독 트리 120그루": pd.Series(single_scores).agg(stats),  # 단독
                     "6그루 다수결 20회": pd.Series(ensemble_scores).agg(stats)}).T  # 다수결
print(summ.round(2).to_string())                              # 원본 셀 61의 표
print("FINAL_HOLDOUT open count:", FINAL_HOLDOUT.open_count)  # 여전히 1


# 4단계: 세 모델을 같은 CV 계약으로 회고 비교 (선택은 바꾸지 않음)
# 왜 · 이미 고른 모델을 다른 모델과 같은 조건에서 돌아보며 비교하기 위해

lr3000 = LogisticRegression(max_iter=3000)                           # 원본 설정
trio = {"LogisticRegression(+Scaler)": make_pipeline(StandardScaler(), lr3000),  # 조건 맞추기
        "KNN k=5 (+Scaler)": make_pipeline(StandardScaler(), KNeighborsClassifier()),
        "DecisionTree(max_depth=3)": DT(max_depth=3, random_state=RS)}
for name, m in trio.items():                                         # 같은 개발 120 · 같은 CV
    c, mean, std = cv_report(m, Xd, yd)                              # 원본 셀 64
    print(f"{name:<28s} {c}/120 · mean {mean:.4f} · std {std:.4f}")  # 세 줄
# 규율 확인
print("선택 변경 없음:", best_dt is gs.best_estimator_, "· open count:", FINAL_HOLDOUT.open_count)

               mean   std    min    max
단독 트리 120그루  116.27  1.52  111.0  119.0
6그루 다수결 20회  117.35  0.81  116.0  119.0
FINAL_HOLDOUT open count: 1


LogisticRegression(+Scaler)  115/120 · mean 0.9583 · std 0.0264
KNN k=5 (+Scaler)            115/120 · mean 0.9583 · std 0.0264
DecisionTree(max_depth=3)    115/120 · mean 0.9583 · std 0.0264
선택 변경 없음: True · open count: 1
